# IntelliFold: Controllable Foundation Model for Biomolecular Structure Prediction

## CSV Batch Processor for High-Throughput Predictions

Easy-to-use batch processing interface for [IntelliFold](https://github.com/IntelliGen-AI/IntelliFold), a controllable foundation model for general and specialized biomolecular structure prediction. IntelliFold delivers state-of-the-art accuracy comparable to AlphaFold3, with comprehensive support for proteins, DNA, RNA, and ligand complexes.

### **Key Features:**
- **Batch Processing**: CSV-based workflow for processing hundreds of predictions efficiently
- **State-of-the-Art Accuracy**: Performance comparable to AlphaFold3 across diverse biomolecular systems
- **Comprehensive Support**: Proteins, DNA, RNA, ligands (CCD codes and SMILES), ions, glycans
- **Post-Translational Modifications**: Built-in support for 15 common PTMs plus custom modifications
- **Advanced Constraints**: Pocket binding constraints and covalent bond specifications
- **Automatic MSA Generation**: Integrated ColabFold MSA server for protein sequences
- **Google Drive Integration**: Automatic upload of results to your Drive
- **Open Source**: Apache 2.0 license for academic and commercial use

### **Workflow:**
1. **Prepare CSV**: Create input file with sequences, modifications, and constraints
2. **Configure**: Set prediction parameters (recycling, sampling, precision)
3. **Run Batch**: Automated processing with progress tracking
4. **Download**: Results automatically uploaded to Google Drive

### **CSV Input Format:**
- **Required columns**: `jobname`, `seq1_name`, `seq1_type`, `seq1`
- **Optional columns**: `seq1_copies`, `seq1_mods`, `pocket_binder`, `pocket_contacts`, `covalent_bonds`
- **Supports**: Up to 10 sequences per job (seq1 through seq10)
- **Sequence types**: protein, dna, rna, ccd (ligand), smiles (ligand)

### **Supported Modifications:**
- **PTMs**: Phosphorylation (SEP, TPO, PTR), Methylation (MLY, M3L), Acetylation (ALY), and more
- **Ligands**: ATP, GTP, NAD, FAD, SAM, Heme, and 20+ common molecules
- **Ions**: Ca²⁺, Mg²⁺, Zn²⁺, Fe²⁺/³⁺, and other metal ions
- **Glycans**: NAG, MAN, GAL, FUC, and other carbohydrate modifications
- **DNA/RNA Mods**: Methylation, oxidation, and other nucleotide modifications
- **Custom**: Upload your own reference CSV with additional modifications

### **Repository:**
- [IntelliFold GitHub Repository](https://github.com/IntelliGen-AI/IntelliFold)
- [IntelliFold Web Server](https://server.intfold.com) - No installation required
- [IntelliFold PyPI Package](https://pypi.org/project/intellifold/)

### **Citations:**

**IntelliFold:**

[The IntFold Team, Qiao L, Bai W, et al. IntFold: A Controllable Foundation Model for General and Specialized Biomolecular Structure Prediction. *arXiv*, 2025](https://arxiv.org/abs/2507.02025)

**If using automatic MSA generation:**

[Mirdita M, Schütze K, Moriwaki Y, et al. ColabFold: making protein folding accessible to all. *Nature Methods*, 2022](https://doi.org/10.1038/s41592-022-01488-1)

---

### **Known Limitations:**
- **Ligands (CCD codes)**: IntelliFold v2.0.0 may trigger `Boost.Python.ArgumentError` in rdkit when processing ligands specified as CCD codes (e.g., ATP, ADP). This appears to be a bug in IntelliFold's rdkit-based CCD preprocessing. The notebook automatically converts known CCD codes to SMILES strings to avoid this, but rare or custom CCD codes not in the built-in reference table may still fail. **Workaround**: Use direct SMILES strings in your CSV instead of CCD codes for ligands not in the reference table. Jobs without ligands (protein-only, protein-DNA, protein-RNA, protein-ion) are unaffected.

### **Quick Start Guide:**

1. **Run Cell 1**: Kernel restart (if needed — skips if already installed)
2. **Run Cell 2**: Install IntelliFold (skips if already installed)
3. **Run Cell 3**: Initialize CSV processor
4. **Run Cell 4**: Upload your CSV file and connect Google Drive
5. **Run Cell 5**: Configure prediction parameters
6. **Run Cell 6**: Execute batch predictions (results auto-uploaded to Drive)

**Example CSV structure:**
```csv
jobname,seq1_name,seq1_type,seq1,seq2_name,seq2_type,seq2
my_protein,proteinA,protein,MSEQVENCE...,ligandATP,ccd,ATP
complex_with_ptm,proteinB,protein,MKLLVV...,,,
```

For proteins with PTMs: `seq1_mods` column with format `POSITION:CCD` (e.g., `10:SEP,25:PTR`)

---

**Ready to start? Run Cell 1 below!** ⬇️

In [ ]:
#@title Cell 1: Kernel Restart (if needed)
#@markdown **Run this first.** Restarts the kernel to clear NumPy 2.x from memory.
#@markdown Skips automatically if IntelliFold is already installed.
import subprocess, sys, os, time

restart_marker = "/content/.intellifold_numpy_restart"
_need_restart = True

# Check if IntelliFold is already installed with correct NumPy
_check = subprocess.run(
    [sys.executable, "-c", "import intellifold; import numpy; assert numpy.__version__.startswith('1.26'); print('OK')"],
    capture_output=True, text=True, timeout=10
)
if _check.returncode == 0 and 'OK' in _check.stdout:
    _need_restart = False
    if os.path.exists(restart_marker):
        os.remove(restart_marker)
    print("=" * 60)
    print(f"{chr(0x2705)} ALREADY INSTALLED -- SKIPPING RESTART")
    print("=" * 60)
    print("IntelliFold + NumPy 1.26.x detected.")
    print("Proceed to Cell 2 (it will also skip).")

elif os.path.exists(restart_marker):
    _need_restart = False
    print("=" * 60)
    print(f"{chr(0x2705)} KERNEL RESTART SUCCESSFUL")
    print("=" * 60)
    print("NumPy 2.x cleared from memory.")
    print()
    print(f"{chr(0x27a1)}  Now run Cell 2 to install IntelliFold.")
    print("=" * 60)

if _need_restart:
    _r = subprocess.run(
        [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
        capture_output=True, text=True
    )
    numpy_ver = _r.stdout.strip() if _r.returncode == 0 else "unknown"

    _r2 = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    gpu_info = _r2.stdout.strip() if _r2.returncode == 0 else "unknown"

    print("=" * 60)
    print(f"{chr(0x1f504)} RESTARTING KERNEL")
    print("=" * 60)
    print(f"   GPU: {gpu_info}")
    print(f"   Current NumPy: {numpy_ver} (need 1.26.x)")
    print(f"   Reason: Colab pre-loads NumPy 2.x into memory.")
    print(f"           Must restart to clear it before installing 1.26.")
    print()
    print(f"   {chr(0x23f3)} Kernel will restart in 3 seconds...")
    print(f"   {chr(0x27a1)}  After restart, run Cell 2 to install.")
    print("=" * 60)
    sys.stdout.flush()

    with open(restart_marker, "w") as f:
        f.write("restarted")

    time.sleep(3)
    os.kill(os.getpid(), 9)


In [ ]:
#@title Cell 2: Upload CSV and Connect Google Drive
#@markdown Upload your CSV file and connect Google Drive. Then click **Run All Below** for hands-free execution.
%%time

import subprocess
import sys
import os

# Quick-install pydrive2 (not in Colab baseline)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydrive2"], capture_output=True, text=True)

from google.colab import files

# Configuration options
setup_google_drive = True #@param {type:"boolean"}
#@markdown - Setup Google Drive for automatic result upload

gdrive_folder_name = "IntelliFold_Results" #@param {type:"string"}
#@markdown - Google Drive folder name for batch results

msa_folder = ""  #@param {type:"string"}
#@markdown > **Pre-computed MSAs** (from MSA Colab). Leave empty for online MSA.
#@markdown > Enter a path, zip filename, or Google Drive zip name. The notebook will
#@markdown > resolve it to the `paired/` directory automatically.

print("=" * 60)
print(f"{chr(0x1f4c1)} UPLOAD CSV AND CONNECT GOOGLE DRIVE")
print("=" * 60)

print(f"\n{chr(0x1f4ca)} Upload your input CSV file with job specifications")
print("Required columns: jobname, seq1_name, seq1_type, seq1")
print("Optional: seq1_copies, seq1_mods, pocket_binder, pocket_contacts, covalent_bonds")
print("Supports up to 10 sequences per job (seq1 through seq10)")

uploaded_csv = files.upload()

if not uploaded_csv:
    raise ValueError("No CSV file uploaded")

csv_filename = list(uploaded_csv.keys())[0]
print(f"\n{chr(0x2705)} Uploaded: {csv_filename}")

# Setup Google Drive
drive = None
if setup_google_drive:
    try:
        from pydrive2.drive import GoogleDrive
        from pydrive2.auth import GoogleAuth
        from google.colab import auth
        from oauth2client.client import GoogleCredentials

        print(f"\n{chr(0x2601)}  Setting up Google Drive...")
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)
        print(f"{chr(0x2705)} Google Drive connected")
    except Exception as e:
        print(f"{chr(0x26a0)}  Google Drive setup failed: {e}")
        drive = None

# ============================================================
# RESOLVE PRE-COMPUTED MSA FOLDER
# ============================================================
import os as _os

def resolve_msa_folder(msa_folder_input, _drive=None):
    """Resolve user's msa_folder input to the actual paired/ directory path."""
    if not msa_folder_input or not msa_folder_input.strip():
        return ''

    name = msa_folder_input.strip().rstrip('/')

    if name.startswith('/'):
        base_path = name
    else:
        base_path = f"/content/{name}"

    if _os.path.isdir(base_path):
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired
        if _os.path.basename(base_path) == 'paired' and _os.listdir(base_path):
            print(f"   MSA folder resolved: {base_path}")
            return base_path

    zip_path = f"{base_path}.zip"
    if not _os.path.isfile(zip_path):
        zip_path = f"/content/{_os.path.basename(name)}.zip"

    if _os.path.isfile(zip_path):
        print(f"   Found local zip: {zip_path}")
        import zipfile as _zf
        extract_to = _os.path.dirname(zip_path) or '/content'
        with _zf.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_to)
        print(f"   Unzipped to {extract_to}")
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired

    if _drive:
        zip_filename = f"{_os.path.basename(name)}.zip"
        print(f"   Searching Google Drive for {zip_filename}...")
        try:
            file_list = _drive.ListFile({
                'q': f"title='{zip_filename}' and trashed=false"
            }).GetList()
            if file_list:
                gdrive_file = file_list[0]
                local_zip = f"/content/{zip_filename}"
                print(f"   Downloading {zip_filename} from Google Drive...")
                gdrive_file.GetContentFile(local_zip)
                print(f"   Downloaded ({_os.path.getsize(local_zip) / 1024 / 1024:.1f} MB)")

                import zipfile as _zf
                with _zf.ZipFile(local_zip, 'r') as zf:
                    zf.extractall('/content')
                print(f"   Unzipped to /content/")

                paired = _os.path.join(base_path, 'paired')
                if _os.path.isdir(paired) and _os.listdir(paired):
                    print(f"   MSA folder resolved: {paired}")
                    return paired
            else:
                print(f"   {zip_filename} not found on Google Drive")
        except Exception as e:
            print(f"   Google Drive download failed: {e}")

    print(f"   WARNING: Could not resolve MSA folder '{msa_folder_input}'")
    return ''

resolved_msa_folder = ''
if msa_folder:
    print(f"\nResolving MSA folder: {msa_folder}")
    resolved_msa_folder = resolve_msa_folder(msa_folder, drive)
    if resolved_msa_folder:
        print(f"   Using pre-computed MSAs from: {resolved_msa_folder}")
        msa_files = [f for f in _os.listdir(resolved_msa_folder) if f.endswith('_paired.a3m')]
        print(f"   Found {len(msa_files)} paired MSA files")

# Store settings (processor and jobs added in Cell 4)
global_settings = {
    'csv_filename': csv_filename,
    'drive': drive,
    'gdrive_folder_name': gdrive_folder_name,
    'msa_folder': resolved_msa_folder,
}

print("\n" + "=" * 60)
print(f"{chr(0x2705)} CSV uploaded, Google Drive {'connected' if drive else 'skipped'}.")
print("Run all cells below for hands-free install + predict.")
print("=" * 60)

In [ ]:
#@title Cell 3: Install IntelliFold
#@markdown Installs IntelliFold with pinned dependencies. Safe to re-run -- skips if already installed.
%%time
import subprocess
import sys
import os
import re

# Check if NumPy 2.x is still loaded (user skipped Cell 1)
_np_check = subprocess.run(
    [sys.executable, "-c", "import numpy; v=numpy.__version__; print(v); exit(0 if v.startswith('1.') else 1)"],
    capture_output=True, text=True
)
if _np_check.returncode != 0:
    restart_marker = "/content/.intellifold_numpy_restart"
    if not os.path.exists(restart_marker):
        print(f"{chr(0x274c)} NumPy 2.x detected. Please run Cell 1 first to restart the kernel.")
        raise RuntimeError("Run Cell 1 first -- kernel restart needed to clear NumPy 2.x")

# Skip if already installed
try:
    result = subprocess.run(
        [sys.executable, "-c", "import intellifold; import numpy; assert numpy.__version__.startswith('1.26'); print('OK')"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0 and 'OK' in result.stdout:
        print("=" * 60)
        print(f"{chr(0x2705)} ALREADY INSTALLED -- NOTHING TO DO")
        print("=" * 60)
        print("IntelliFold + NumPy 1.26.x verified.")
        print("Proceed to Cell 3.")
        with open("/content/INTELLIFOLD_READY", "w") as f:
            f.write("Ready")
    else:
        raise Exception("Need install")
except RuntimeError:
    raise
except:
    def run_cmd(cmd, desc):
        """Execute command with output suppression unless error"""
        print(f"[{desc}]")
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"FAILED: {result.stderr[:300]}")
            return False
        print("OK")
        return True

    def get_cuda_version():
        """Detect CUDA version from nvidia-smi"""
        try:
            result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
            if result.returncode == 0:
                match = re.search(r'CUDA Version: (\d+\.\d+)', result.stdout)
                if match:
                    version = match.group(1)
                    major = int(version.split('.')[0])
                    minor = int(version.split('.')[1])
                    return major, minor, version
        except Exception as e:
            print(f"{chr(0x26a0)}  Could not detect CUDA version: {e}")
        return None, None, None

    print("=" * 60)
    print("INSTALLING INTELLIFOLD")
    print("=" * 60)

    # Detect CUDA version
    cuda_major, cuda_minor, cuda_version = get_cuda_version()

    if cuda_major is None:
        print(f"{chr(0x274c)} No CUDA detected - cannot proceed")
        sys.exit(1)

    print(f"{chr(0x2705)} CUDA Version: {cuda_version}")
    print(f"   Driver CUDA: {cuda_major}.{cuda_minor}")

    # Check GPU type
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          capture_output=True, text=True)
    if result.returncode == 0:
        gpu_name = result.stdout.strip()
        print(f"{chr(0x2705)} GPU: {gpu_name}")

    # PyTorch version (pip resolves CUDA-compatible wheel automatically)
    pytorch_version = "2.6.0"
    print(f"\n{chr(0x1f4e6)} PyTorch: {pytorch_version} (pip resolves CUDA wheel automatically)")

    # Clean existing installations
    print("\n" + "=" * 60)
    print("[Cleanup]")
    cleanup_packages = [
        'torch', 'torchvision', 'torchaudio',
        'pytorch-lightning', 'torchmetrics',
        'intellifold', 'numpy', 'pandas'
    ]
    subprocess.run(
        f"{sys.executable} -m pip uninstall {' '.join(cleanup_packages)} -y",
        shell=True, capture_output=True
    )
    print("OK")

    # Layer 2: sitecustomize.py (do BEFORE installing numpy)
    sitecustomize_content = """# Colab NumPy priority fix
import sys
import os

if '/env/python' in sys.path:
    sys.path.remove('/env/python')

if 'PYTHONPATH' in os.environ:
    del os.environ['PYTHONPATH']
"""
    sitecustomize_path = "/usr/local/lib/python3.12/dist-packages/sitecustomize.py"
    with open(sitecustomize_path, "w") as f:
        f.write(sitecustomize_content)
    print(f"{chr(0x2705)} sys.path fix installed")

    # Layer 3: Clean current kernel
    if '/env/python' in sys.path:
        sys.path.remove('/env/python')
    if 'PYTHONPATH' in os.environ:
        del os.environ['PYTHONPATH']
    for mod in [k for k in list(sys.modules.keys()) if k.startswith(('numpy', 'pandas', 'np', 'pd'))]:
        del sys.modules[mod]

    # Install NumPy FIRST
    print("\n" + "=" * 60)
    print("[1/7] NumPy 1.26.4")
    if not run_cmd(
        f"{sys.executable} -m pip install --no-cache-dir 'numpy==1.26.4'",
        "Installing numpy==1.26.4"
    ):
        print(f"{chr(0x274c)} NumPy installation failed")
        sys.exit(1)

    result = subprocess.run(
        [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
        capture_output=True, text=True
    )
    if '1.26' not in result.stdout:
        print(f"{chr(0x274c)} NumPy version wrong: {result.stdout.strip()}")
        sys.exit(1)
    print(f"   {chr(0x2705)} Verified: NumPy {result.stdout.strip()}")

    # Install PyTorch
    print("\n" + "=" * 60)
    print(f"[2/7] PyTorch {pytorch_version}")
    if not run_cmd(
        f"{sys.executable} -m pip install torch=={pytorch_version} torchvision torchaudio",
        f"Installing PyTorch {pytorch_version}"
    ):
        print(f"{chr(0x274c)} PyTorch installation failed")
        sys.exit(1)

    # Install pytorch-lightning
    print("\n" + "=" * 60)
    print("[3/7] Lightning stack")
    if not run_cmd(
        f"{sys.executable} -m pip install pytorch-lightning==2.5.0 torchmetrics==1.4.0",
        "Installing Lightning"
    ):
        print(f"{chr(0x274c)} Lightning installation failed")
        sys.exit(1)

    # Install IntelliFold dependencies
    print("\n" + "=" * 60)
    print("[4/7] IntelliFold core dependencies")
    intellifold_deps = [
        "pandas==2.2.3",
        "scipy==1.14.1",
        "biopython==1.85",
        "PyYAML==6.0.2"
    ]
    if not run_cmd(
        f"{sys.executable} -m pip install {' '.join(intellifold_deps)}",
        "Installing core dependencies"
    ):
        print(f"{chr(0x274c)} Core dependencies installation failed")
        sys.exit(1)

    # Install extended dependencies
    print("\n" + "=" * 60)
    print("[5/7] IntelliFold extended dependencies")
    extended_deps = [
        "accelerate==1.1.1",
        "deepspeed==0.16.4",
        "einops==0.8.0",
        "einx==0.3.0",
        "ihm==2.5",
        "mashumaro==3.14",
        "ml_collections==1.0.0",
        "modelcif==1.2",
        "numba==0.61.0",
        "rdkit==2024.3.2",
        "torchdiffeq==0.2.5",
        "tqdm==4.67.1",
        "click==8.1.8",
        "requests==2.32.3",
        "networkx==3.4.2"
    ]
    if not run_cmd(
        f"{sys.executable} -m pip install {' '.join(extended_deps)}",
        "Installing extended dependencies"
    ):
        print(f"{chr(0x26a0)}  Extended dependencies installation failed (may not be critical)")

    # Install IntelliFold itself with --no-deps
    print("\n" + "=" * 60)
    print("[6/7] IntelliFold (--no-deps)")
    if not run_cmd(
        f"{sys.executable} -m pip install intellifold --no-deps",
        "Installing IntelliFold"
    ):
        print(f"{chr(0x274c)} IntelliFold installation failed")
        sys.exit(1)

    # IPython startup fix
    ipython_startup_dir = "/root/.ipython/profile_default/startup"
    os.makedirs(ipython_startup_dir, exist_ok=True)
    with open(os.path.join(ipython_startup_dir, "00-fix_syspath.py"), "w") as f:
        f.write(sitecustomize_content)


# Step 7: Install ipSAE_batch for interface analysis
print("\n" + "=" * 60)
print("[7/7] ipSAE_batch (interface scoring and visualization)")
run_cmd(
    f"{sys.executable} -m pip install seaborn pycirclize python-igraph plotly",
    "Installing visualization dependencies"
)
if not run_cmd(
    f"{sys.executable} -m pip install git+https://github.com/JKourelis/ipSAE_batch.git",
    "Installing ipSAE_batch"
):
    print("WARNING: ipSAE_batch failed to install. Interface analysis will be unavailable.")

    # VERIFICATION
    print("\n" + "=" * 60)
    print("VERIFICATION")
    print("=" * 60)

    print("\n[Testing NumPy]")
    try:
        import numpy as np
        print(f"   {chr(0x2705)} NumPy {np.__version__}")
        if not np.__version__.startswith('1.26'):
            print(f"   {chr(0x274c)} NumPy version wrong: expected 1.26.x, got {np.__version__}")
            sys.exit(1)
    except Exception as e:
        print(f"   {chr(0x274c)} NumPy import failed: {e}")
        sys.exit(1)

    print("\n[Checking packages]")
    try:
        import pandas as pd
        print(f"   {chr(0x2705)} Pandas {pd.__version__}")
    except ImportError:
        print(f"   {chr(0x26a0)}  Pandas not installed")


    # Test ipSAE_batch
    result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True)
    if result.returncode == 0:
        print("   ipsae-batch CLI: OK")
    else:
        print("   ipsae-batch CLI: NOT AVAILABLE (interface analysis will be skipped)")


    # ============================================================
    # ENVIRONMENT & VERSION LOG
    # ============================================================
    env_lines = []
    env_lines.append("\n" + "=" * 60)
    env_lines.append("ENVIRONMENT & INSTALLED VERSIONS")
    env_lines.append("=" * 60)
    
    # Python version
    env_lines.append(f"\nPython: {sys.version.split()[0]}")
    
    # GPU info
    gpu_result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    if gpu_result.returncode == 0:
        env_lines.append(f"GPU: {gpu_result.stdout.strip()}")
    env_lines.append(f"CUDA (driver max): {cuda_version}")
    
    # PyTorch + CUDA runtime
    result = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(f'PyTorch: {torch.__version__}'); "
         "print(f'CUDA available: {torch.cuda.is_available()}'); "
         "print(f'CUDA runtime: {torch.version.cuda}') if torch.cuda.is_available() else None"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        for line in result.stdout.strip().split('\n'):
            env_lines.append(f"   {line}")
    
    # All relevant package versions via pip
    result = subprocess.run(
        [sys.executable, "-m", "pip", "list", "--format=freeze"],
        capture_output=True, text=True
    )
    all_packages = result.stdout.strip().split('\n')
    
    # IntelliFold + core deps
    env_lines.append("\nIntelliFold stack:")
    tool_pkgs = [
        'intellifold',
        'numpy',
        'torch',
        'torchvision',
        'torchaudio',
        'pytorch-lightning',
        'torchmetrics',
        'accelerate',
        'biopython',
        'click',
        'einops',
        'einx',
        'ml-collections',
        'networkx',
        'numba',
        'pandas',
        'rdkit',
        'scipy',
        'tqdm',
    ]
    for pkg in tool_pkgs:
        pkg_lower = pkg.lower().replace('-', '_')
        for line in all_packages:
            line_lower = line.lower().replace('-', '_')
            if line_lower.startswith(pkg_lower + '=='):
                env_lines.append(f"   {line}")
                break
        else:
            # Try partial match
            for line in all_packages:
                if pkg_lower in line.lower().replace('-', '_'):
                    env_lines.append(f"   {line}")
                    break
    
    # ipSAE_batch + visualization deps
    env_lines.append("\nipSAE_batch stack:")
    ipsae_pkgs = [
        'ipsae-batch', 'seaborn', 'pycirclize', 'python-igraph', 'plotly',
        'matplotlib', 'networkx',
    ]
    for pkg in ipsae_pkgs:
        pkg_lower = pkg.lower().replace('-', '_')
        for line in all_packages:
            line_lower = line.lower().replace('-', '_')
            if line_lower.startswith(pkg_lower + '=='):
                env_lines.append(f"   {line}")
                break
        else:
            for line in all_packages:
                if pkg_lower in line.lower().replace('-', '_'):
                    env_lines.append(f"   {line}")
                    break
            else:
                env_lines.append(f"   {pkg}: not installed")

    # Cleanup restart marker and mark ready
    restart_marker = "/content/.intellifold_numpy_restart"
    if os.path.exists(restart_marker):
        os.remove(restart_marker)
    with open("/content/INTELLIFOLD_READY", "w") as f:
        f.write("Ready")

    env_lines.append("\n" + "=" * 60)

    # Write environment log to file and print
    env_log = "\n".join(str(x) for x in env_lines)
    print(env_log)
    with open("/content/environment.txt", "w") as _ef:
        _ef.write(env_log + "\n")
    print(f"{chr(0x2705)} INTELLIFOLD INSTALLATION COMPLETE")
    print("=" * 60)
    print("Next: Run Cell 3 to set up the job processor")


In [ ]:
#@title Cell 4: IntelliFold CSV Processor Setup
import pandas as pd
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from io import StringIO

class IntelliFoldJobProcessor:
    """Process CSV files for IntelliFold batch predictions"""

    EMBEDDED_REFERENCE = """Type,CCD Code,Name,Target Residue,Molecular Formula,All Atom Count,Heavy Atom Count,SMILES
PTM,SEP,Phosphoserine,SER,C3H8NO6P,18,11
PTM,TPO,Phosphothreonine,THR,C4H10NO6P,21,12
PTM,PTR,Phosphotyrosine,TYR,C9H12NO6P,28,17
PTM,MLY,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,ALY,N-Acetyllysine,LYS,C8H16N2O3,29,13
PTM,HYP,Hydroxyproline,PRO,C5H9NO3,18,9
PTM,M3L,N-Trimethyllysine,LYS,C9H20N2O2,33,13
PTM,MLZ,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,CSD,Cysteine sulfinic acid,CYS,C3H7NO4S,16,9
PTM,CSO,S-Hydroxycysteine,CYS,C3H7NO3S,15,8
PTM,CGU,Gamma-carboxyglutamic acid,GLU,C5H7NO6,21,12
PTM,FME,N-Formylmethionine,MET,C6H11NO3S,22,11
PTM,NEP,N-(phosphonoethyl)isoleucine,ILE,C8H18NO5P,32,15
PTM,HIC,4-Methyl-histidine,HIS,C7H11N3O2,23,12
PTM,CAS,S-(dimethylarsenic)cysteine,CYS,C5H11AsNO2S,20,9
Ligand,ATP,Adenosine triphosphate,NA,C10H16N5O13P3,47,31,Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P@](O)(=O)O[P@@](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,ADP,Adenosine diphosphate,NA,C10H15N5O10P2,42,27,Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P@@](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,AMP,Adenosine monophosphate,NA,C10H14N5O7P,37,23,Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,GTP,Guanosine triphosphate,NA,C10H16N5O14P3,48,32,NC1=Nc2n(cnc2C(=O)N1)[C@@H]3O[C@H](CO[P](O)(=O)O[P](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,GDP,Guanosine diphosphate,NA,C10H15N5O11P2,43,28,NC1=Nc2n(cnc2C(=O)N1)[C@@H]3O[C@H](CO[P](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,GMP,Guanosine monophosphate,NA,C10H14N5O8P,38,24,NC1=Nc2n(cnc2C(=O)N1)[C@@H]3O[C@H](CO[P](O)(O)=O)[C@@H](O)[C@H]3O
Ligand,CTP,Cytidine triphosphate,NA,C9H16N3O14P3,45,29,NC1=NC(=O)N(C=C1)[C@@H]2O[C@H](CO[P@@](O)(=O)O[P@@](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]2O
Ligand,CDP,Cytidine diphosphate,NA,C9H15N3O11P2,40,25,NC1=NC(=O)N(C=C1)[C@@H]2O[C@H](CO[P@](O)(=O)O[P](O)(O)=O)[C@@H](O)[C@H]2O
Ligand,UTP,Uridine triphosphate,NA,C9H15N2O15P3,44,29,O[C@H]1[C@@H](O)[C@@H](O[C@@H]1CO[P@](O)(=O)O[P@](O)(=O)O[P](O)(O)=O)N2C=CC(=O)NC2=O
Ligand,UDP,Uridine diphosphate,NA,C9H14N2O12P2,39,25,O[C@H]1[C@@H](O)[C@@H](O[C@@H]1CO[P](O)(=O)O[P](O)(O)=O)N2C=CC(=O)NC2=O
Ligand,NAD,Nicotinamide adenine dinucleotide,NA,C21H27N7O14P2,71,44,NC(=O)c1ccc[n+](c1)[C@@H]2O[C@H](CO[P]([O-])(=O)O[P@](O)(=O)OC[C@H]3O[C@H]([C@H](O)[C@@H]3O)n4cnc5c(N)ncnc45)[C@@H](O)[C@H]2O
Ligand,NAP,NADP,NA,C21H28N7O17P3,86,55,NC(=O)c1ccc[n+](c1)[C@@H]2O[C@H](CO[P]([O-])(=O)O[P@@](O)(=O)OC[C@H]3O[C@H]([C@H](O[P](O)(O)=O)[C@@H]3O)n4cnc5c(N)ncnc45)[C@@H](O)[C@H]2O
Ligand,FAD,Flavin adenine dinucleotide,NA,C27H33N9O15P2,91,53,Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P@](O)(=O)O[P@@](O)(=O)OC[C@H]4O[C@H]([C@H](O)[C@@H]4O)n5cnc6c(N)ncnc56)c2cc1C
Ligand,FMN,Flavin mononucleotide,NA,C17H21N4O9P,52,31,Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P](O)(O)=O)c2cc1C
Ligand,HEM,Heme,NA,C34H32FeN4O4,75,43,CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C
Ligand,SAH,S-Adenosyl-L-homocysteine,NA,C14H20N6O5S,46,26,N[C@@H](CCSC[C@H]1O[C@H]([C@H](O)[C@@H]1O)n2cnc3c(N)ncnc23)C(O)=O
Ligand,SAM,S-Adenosyl-L-methionine,NA,C15H22N6O5S,49,27,C[S@@+](CC[C@H](N)C([O-])=O)C[C@H]1O[C@H]([C@H](O)[C@@H]1O)n2cnc3c(N)ncnc23
Ligand,COA,Coenzyme A,NA,C21H36N7O16P3S,90,57,CC(C)(CO[P@@](O)(=O)O[P@](O)(=O)OC[C@H]1O[C@H]([C@H](O)[C@@H]1O[P](O)(O)=O)n2cnc3c(N)ncnc23)[C@@H](O)C(=O)NCCC(=O)NCCS
Ligand,ACO,Acetyl coenzyme A,NA,C23H38N7O17P3S,99,61,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)CO[P@@](O)(=O)O[P@](O)(=O)OC[C@H]1O[C@H]([C@H](O)[C@@H]1O[P](O)(O)=O)n2cnc3c(N)ncnc23
Ligand,PLP,Pyridoxal-5-phosphate,NA,C8H10NO6P,25,16,Cc1ncc(CO[P](O)(O)=O)c(C=O)c1O
Ligand,TPP,Thiamine diphosphate,NA,C12H19N4O7P2S,45,25,Cc1ncc(C[n+]2csc(CCO[P@@](O)(=O)O[P](O)(O)=O)c2C)c(N)n1
Ligand,BTN,Biotin,NA,C10H16N2O3S,32,16,OC(=O)CCCC[C@@H]1SC[C@@H]2NC(=O)N[C@H]12
Ligand,MTA,5-Methylthioadenosine,NA,C11H15N5O3S,35,20,CSC[C@H]1O[C@H]([C@H](O)[C@@H]1O)n2cnc3c(N)ncnc23
Ligand,THM,Thiamine,NA,C12H17ClN4OS,38,18,
Ion,MG,Magnesium ion,NA,Mg,1,1
Ion,ZN,Zinc ion,NA,Zn,1,1
Ion,CA,Calcium ion,NA,Ca,1,1
Ion,FE,Iron ion,NA,Fe,1,1
Ion,MN,Manganese ion,NA,Mn,1,1
Ion,CU,Copper ion,NA,Cu,1,1
Ion,CO,Cobalt ion,NA,Co,1,1
Ion,NI,Nickel ion,NA,Ni,1,1
Ion,K,Potassium ion,NA,K,1,1
Ion,NA,Sodium ion,NA,Na,1,1
Ion,CL,Chloride ion,NA,Cl,1,1
Glycan,NAG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,MAN,D-Mannose,NA,C6H12O6,24,12
Glycan,FUC,L-Fucose,NA,C6H12O5,23,11
Glycan,GAL,D-Galactose,NA,C6H12O6,24,12
Glycan,SIA,N-Acetylneuraminic acid,NA,C11H19NO9,40,21
Glycan,GLC,D-Glucose,NA,C6H12O6,24,12
Glycan,BMA,beta-D-Mannose,NA,C6H12O6,24,12
Glycan,NDG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,A2G,N-Acetyl-D-galactosamine,NA,C8H15NO6,30,15
Glycan,FUL,L-Fucose,NA,C6H12O5,23,11
DNA_Mod,5MC,5-Methylcytosine,DC,C10H15N3O7P,36,21
DNA_Mod,6MA,N6-Methyladenine,DA,C11H16N5O6P,39,23
DNA_Mod,5HMC,5-Hydroxymethylcytosine,DC,C10H15N3O8P,37,22
DNA_Mod,8OG,8-Oxoguanine,DG,C10H13N5O8P,37,24
DNA_Mod,M2G,N2-Methylguanine,DG,C11H16N5O7P,40,24
DNA_Mod,4MC,N4-Methylcytosine,DC,C10H15N3O7P,36,21
DNA_Mod,1MA,1-Methyladenine,DA,C11H16N5O6P,39,23
DNA_Mod,3MA,3-Methyladenine,DA,C11H16N5O6P,39,23
RNA_Mod,PSU,Pseudouridine,U,C9H12N2O9P,33,21
RNA_Mod,1MA,1-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,7MG,7-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,5MC,5-Methylcytidine,C,C10H15N3O8P,37,22
RNA_Mod,2MA,2-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,M2G,N2-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,M7G,7-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,M1A,1-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,OMC,2'-O-Methylcytidine,C,C10H15N3O8P,37,22
RNA_Mod,OMG,2'-O-Methylguanosine,G,C11H15N5O8P,40,25"""

    def __init__(self, reference_csv: Optional[str] = None):
        """Initialize processor with reference data"""
        if reference_csv:
            self.reference_data = pd.read_csv(StringIO(reference_csv))
        else:
            self.reference_data = pd.read_csv(StringIO(self.EMBEDDED_REFERENCE))

        self.ptm_lookup = self._create_lookup('PTM')
        self.ligand_lookup = self._create_lookup('Ligand')
        self.ion_lookup = self._create_lookup('Ion')
        self.glycan_lookup = self._create_lookup('Glycan')
        self.dna_mod_lookup = self._create_lookup('DNA_Mod')
        self.rna_mod_lookup = self._create_lookup('RNA_Mod')

        self.aa_3to1 = {
            'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
            'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
            'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
            'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
        }

        self.amino_acids = set('ACDEFGHIKLMNPQRSTVWY')

    def _create_lookup(self, type_name: str) -> Dict[str, Dict[str, Any]]:
        """Create lookup dictionary for a specific type"""
        type_data = self.reference_data[self.reference_data['Type'] == type_name]
        lookup = {}
        for _, row in type_data.iterrows():
            if pd.notna(row['CCD Code']):
                entry = {
                    'name': row['Name'],
                    'target_residue': row['Target Residue'] if pd.notna(row['Target Residue']) else 'NA',
                    'smiles': row['SMILES'] if 'SMILES' in row.index and pd.notna(row.get('SMILES')) else None
                }
                if 'SMILES' in row and pd.notna(row.get('SMILES')) and str(row.get('SMILES', '')).strip():
                    entry['smiles'] = str(row['SMILES']).strip()
                lookup[row['CCD Code']] = entry
        return lookup

    def _validate_sequence_characters(self, sequence: str, seq_type: str) -> List[str]:
        """Validate sequence contains only allowed characters"""
        errors = []
        sequence = sequence.upper()

        if seq_type.lower() == 'protein':
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in self.amino_acids:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid amino acids: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'dna':
            valid_bases = set('ATCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid DNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'rna':
            valid_bases = set('AUCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid RNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        return errors

    def _is_smiles(self, ligand_string: str) -> bool:
        """Check if string is likely a SMILES representation"""
        if len(ligand_string) < 3:
            return False
        smiles_chars = set('[]()=#@+-\\/CNOPSFClBrI0123456789')
        return any(char in smiles_chars for char in ligand_string)

    def _remap_modification_chains(self, mod_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap modification chain IDs from user names to A, B, C... format"""
        if pd.isna(mod_string) or str(mod_string).strip() == '':
            return mod_string

        mod_string = str(mod_string).strip()

        for old_name, new_chain_id in name_mapping.items():
            mod_string = mod_string.replace(f"{old_name}:", f"{new_chain_id}:")

        return mod_string

    def _remap_contact_chains(self, contacts_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap contact chain IDs from user names to A, B, C... format"""
        if pd.isna(contacts_string) or str(contacts_string).strip() == '':
            return contacts_string

        contacts_string = str(contacts_string).strip()

        for old_name, new_chain_id in name_mapping.items():
            contacts_string = contacts_string.replace(f"{old_name}:", f"{new_chain_id}:")

        return contacts_string

    def _remap_bond_chains(self, bonds_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap covalent bond chain IDs from user names to A, B, C... format"""
        if pd.isna(bonds_string) or str(bonds_string).strip() == '':
            return bonds_string

        bonds_string = str(bonds_string).strip()

        for old_name, new_chain_id in name_mapping.items():
            bonds_string = bonds_string.replace(f"{old_name}:", f"{new_chain_id}:")

        return bonds_string

    def _parse_modifications(self, mod_string: str, sequence: str, chain_id: str, seq_type: str) -> Tuple[List[Dict], List[str]]:
        """Parse modifications in format: CHAIN:POSITION:CCD_CODE"""
        errors = []
        mods = []

        if pd.isna(mod_string) or str(mod_string).strip() == '':
            return mods, errors

        mod_string = str(mod_string).strip()
        mod_entries = [e.strip() for e in mod_string.split(';') if e.strip()]

        if seq_type.lower() == 'protein':
            mod_lookups = [self.ptm_lookup, self.glycan_lookup]
            mod_types = ['PTM', 'Glycan']
        elif seq_type.lower() == 'dna':
            mod_lookups = [self.dna_mod_lookup]
            mod_types = ['DNA Mod']
        elif seq_type.lower() == 'rna':
            mod_lookups = [self.rna_mod_lookup]
            mod_types = ['RNA Mod']
        else:
            return mods, errors

        for entry in mod_entries:
            parts = entry.split(':')
            if len(parts) != 3:
                errors.append(f"Invalid modification format: '{entry}'. Use CHAIN:POSITION:CCD_CODE")
                continue

            mod_chain, pos_str, ccd_code = parts
            mod_chain = mod_chain.strip()
            ccd_code = ccd_code.strip()

            if mod_chain != chain_id:
                continue

            try:
                position = int(pos_str.strip())

                found = False
                for lookup, mod_type in zip(mod_lookups, mod_types):
                    if ccd_code in lookup:
                        found = True

                        if position < 1 or position > len(sequence):
                            errors.append(f"{mod_type} position {position} out of range (sequence length: {len(sequence)})")
                            continue

                        target_residue = lookup[ccd_code]['target_residue']
                        actual_residue = sequence[position - 1].upper()

                        if target_residue != 'NA':
                            if mod_type == 'Glycan':
                                if actual_residue not in ['N', 'T', 'S']:
                                    errors.append(f"Glycan {ccd_code} requires N/T/S but found {actual_residue} at position {position}")
                                    continue
                            elif mod_type == 'PTM':
                                target_1letter = self.aa_3to1.get(target_residue.upper())
                                if target_1letter and actual_residue != target_1letter:
                                    errors.append(f"PTM {ccd_code} targets {target_residue}({target_1letter}) but found {actual_residue} at position {position}")
                                    continue
                            else:
                                if len(target_residue) > 1 and target_residue.startswith('D'):
                                    target_residue = target_residue[1]
                                if actual_residue != target_residue:
                                    errors.append(f"{mod_type} {ccd_code} targets {target_residue} but found {actual_residue} at position {position}")
                                    continue

                        mods.append({
                            'chain_id': mod_chain,
                            'position': position,
                            'ccd': ccd_code
                        })
                        break

                if not found:
                    errors.append(f"Unknown modification code: '{ccd_code}'")

            except ValueError:
                errors.append(f"Invalid position in modification: '{entry}'")

        return mods, errors

    def _parse_pocket(self, binder: str, contacts_string: str) -> Tuple[Optional[Dict], List[str]]:
        """Parse pocket constraint"""
        errors = []

        if pd.isna(binder) or str(binder).strip() == '':
            return None, errors

        binder = str(binder).strip()

        if pd.isna(contacts_string) or str(contacts_string).strip() == '':
            errors.append(f"Pocket binder '{binder}' specified but no contacts provided")
            return None, errors

        contacts_string = str(contacts_string).strip()
        contact_entries = [e.strip() for e in contacts_string.split(';') if e.strip()]

        contacts = []
        for entry in contact_entries:
            parts = entry.split(':')
            if len(parts) != 2:
                errors.append(f"Invalid contact format: '{entry}'. Use CHAIN:RESIDUE")
                continue

            try:
                chain_id, residue_num = parts
                contacts.append([chain_id.strip(), int(residue_num.strip())])
            except ValueError:
                errors.append(f"Invalid contact specification: '{entry}'")

        if not contacts:
            errors.append(f"No valid contacts parsed for pocket binder '{binder}'")
            return None, errors

        return {
            'binder': binder,
            'contacts': contacts
        }, errors

    def _parse_covalent_bonds(self, bonds_string: str) -> Tuple[List[Dict], List[str]]:
        """Parse covalent bond constraints"""
        errors = []
        bonds = []

        if pd.isna(bonds_string) or str(bonds_string).strip() == '':
            return bonds, errors

        bonds_string = str(bonds_string).strip()
        bond_entries = [e.strip() for e in bonds_string.split(';') if e.strip()]

        for entry in bond_entries:
            parts = entry.split(':')
            if len(parts) != 6:
                errors.append(f"Invalid covalent bond format: '{entry}'. Use CHAIN:RES:ATOM:CHAIN:RES:ATOM")
                continue

            try:
                chain1, res1, atom1, chain2, res2, atom2 = parts
                bonds.append({
                    'atom1': [chain1.strip(), int(res1.strip()), atom1.strip()],
                    'atom2': [chain2.strip(), int(res2.strip()), atom2.strip()]
                })
            except ValueError:
                errors.append(f"Invalid covalent bond specification: '{entry}'")

        return bonds, errors

    def _sanitize_jobname(self, jobname: str) -> Tuple[str, List[str]]:
        """Sanitize jobname for filesystem compatibility"""
        errors = []

        if pd.isna(jobname) or str(jobname).strip() == '':
            errors.append("Missing jobname")
            return '', errors

        original = str(jobname)
        sanitized = original.lower()
        sanitized = sanitized.replace('-', '_')
        sanitized = re.sub(r'[^a-z0-9_]', '', sanitized)

        if len(sanitized) > 128:
            sanitized = sanitized[:128]
            errors.append(f"Jobname truncated to 128 characters")

        if not sanitized.strip():
            errors.append(f"Jobname '{original}' became empty after sanitization")
            return '', errors

        return sanitized, errors

    def _generate_yaml(self, job: Dict) -> str:
        """Generate YAML format for IntelliFold"""
        lines = ["version: 1", "sequences:"]

        protein_groups = {}
        dna_groups = {}
        rna_groups = {}
        ligand_groups = {}

        for seq in job['sequences']:
            seq_type = seq['type']

            if seq_type == 'protein':
                key = (seq['sequence'], tuple(sorted((m['position'], m['ccd']) for m in seq['modifications'])) if seq['modifications'] else ())
                if key not in protein_groups:
                    protein_groups[key] = []
                protein_groups[key].append(seq)

            elif seq_type == 'dna':
                key = (seq['sequence'], tuple(sorted((m['position'], m['ccd']) for m in seq['modifications'])) if seq['modifications'] else ())
                if key not in dna_groups:
                    dna_groups[key] = []
                dna_groups[key].append(seq)

            elif seq_type == 'rna':
                key = (seq['sequence'], tuple(sorted((m['position'], m['ccd']) for m in seq['modifications'])) if seq['modifications'] else ())
                if key not in rna_groups:
                    rna_groups[key] = []
                rna_groups[key].append(seq)

            elif seq_type == 'ligand':
                if 'smiles' in seq:
                    key = ('smiles', seq['smiles'])
                else:
                    key = ('ccd', seq['ccd'])
                if key not in ligand_groups:
                    ligand_groups[key] = []
                ligand_groups[key].append(seq)

        for (sequence, mod_tuple), seqs in protein_groups.items():
            lines.append("  - protein:")
            chain_ids = [s['id'] for s in seqs]
            if len(chain_ids) == 1:
                lines.append(f"      id: {chain_ids[0]}")
            else:
                lines.append(f"      id: [{', '.join(chain_ids)}]")
            lines.append(f"      sequence: {sequence}")

            if seqs[0]['modifications']:
                lines.append("      modifications:")
                for mod in seqs[0]['modifications']:
                    lines.append(f"        - ptmType: {mod['ccd']}")
                    lines.append(f"          ptmPosition: {mod['position']}")

        for (sequence, mod_tuple), seqs in dna_groups.items():
            lines.append("  - dna:")
            chain_ids = [s['id'] for s in seqs]
            if len(chain_ids) == 1:
                lines.append(f"      id: {chain_ids[0]}")
            else:
                lines.append(f"      id: [{', '.join(chain_ids)}]")
            lines.append(f"      sequence: {sequence}")

            if seqs[0]['modifications']:
                lines.append("      modifications:")
                for mod in seqs[0]['modifications']:
                    lines.append(f"        - modificationType: {mod['ccd']}")
                    lines.append(f"          basePosition: {mod['position']}")

        for (sequence, mod_tuple), seqs in rna_groups.items():
            lines.append("  - rna:")
            chain_ids = [s['id'] for s in seqs]
            if len(chain_ids) == 1:
                lines.append(f"      id: {chain_ids[0]}")
            else:
                lines.append(f"      id: [{', '.join(chain_ids)}]")
            lines.append(f"      sequence: {sequence}")

            if seqs[0]['modifications']:
                lines.append("      modifications:")
                for mod in seqs[0]['modifications']:
                    lines.append(f"        - modificationType: {mod['ccd']}")
                    lines.append(f"          basePosition: {mod['position']}")

        for (lig_type, lig_value), seqs in ligand_groups.items():
            lines.append("  - ligand:")
            chain_ids = [s['id'] for s in seqs]
            if len(chain_ids) == 1:
                lines.append(f"      id: {chain_ids[0]}")
            else:
                lines.append(f"      id: [{', '.join(chain_ids)}]")

            if lig_type == 'smiles':
                lines.append(f"      smiles: '{lig_value}'")
            else:
                # Use SMILES instead of CCD when available (avoids rdkit CCD parsing bugs)
                known = self.ligand_lookup.get(lig_value, {})
                if known.get('smiles'):
                    lines.append(f"      smiles: '{known['smiles']}'")
                else:
                    lines.append(f"      ccd: {lig_value}")

        if job.get('pocket') or job.get('covalent_bonds'):
            lines.append("constraints:")

            if job.get('pocket'):
                pocket = job['pocket']
                lines.append("  - pocket:")
                lines.append(f"      binder: {pocket['binder']}")
                lines.append("      contacts:")
                for contact in pocket['contacts']:
                    lines.append(f"        - [{contact[0]}, {contact[1]}]")

            if job.get('covalent_bonds'):
                for bond in job['covalent_bonds']:
                    lines.append("  - bond:")
                    lines.append(f"      atom1: [{bond['atom1'][0]}, {bond['atom1'][1]}, {bond['atom1'][2]}]")
                    lines.append(f"      atom2: [{bond['atom2'][0]}, {bond['atom2'][1]}, {bond['atom2'][2]}]")

        return '\n'.join(lines)

    def _process_job(self, row: pd.Series) -> Tuple[Optional[Dict], List[str]]:
        """Process a single job row from CSV"""
        errors = []
        all_sequences = []

        jobname, jobname_errors = self._sanitize_jobname(row.get('jobname', ''))
        errors.extend(jobname_errors)

        if not jobname:
            return None, errors

        pocket_binder = row.get('pocket_binder', '')
        pocket_contacts = row.get('pocket_contacts', '')
        covalent_bonds_str = row.get('covalent_bonds', '')

        chain_id_counter = 0
        name_to_chain_mapping = {}

        for i in range(1, 11):
            name_col = f'seq{i}_name'
            type_col = f'seq{i}_type'
            copies_col = f'seq{i}_copies'
            seq_col = f'seq{i}'
            mods_col = f'seq{i}_mods'

            if name_col not in row or type_col not in row or seq_col not in row:
                continue

            user_name = row.get(name_col, '')
            seq_type = row.get(type_col, '')
            seq_type = str(seq_type).strip().lower()
            copies = row.get(copies_col, 1)
            sequence = row.get(seq_col, '')
            mods = row.get(mods_col, '')

            if pd.isna(sequence) or str(sequence).strip() == '':
                continue

            sequence = str(sequence).strip()
            copies = int(copies) if pd.notna(copies) and str(copies).strip() != '' else 1
            user_name = str(user_name).strip() if pd.notna(user_name) else ''

            chain_ids = []
            for copy_num in range(copies):
                chain_id = chr(ord('A') + chain_id_counter)
                chain_ids.append(chain_id)

                if user_name and copy_num == 0:
                    name_to_chain_mapping[user_name] = chain_id

                chain_id_counter += 1

            if seq_type.lower() in ['protein', 'dna', 'rna']:
                char_errors = self._validate_sequence_characters(sequence, seq_type)
                errors.extend(char_errors)

                for idx, chain_id in enumerate(chain_ids):
                    remapped_mods = self._remap_modification_chains(mods, name_to_chain_mapping)
                    mods_list, mod_errors = self._parse_modifications(remapped_mods, sequence, chain_id, seq_type)
                    errors.extend(mod_errors)

                    seq_dict = {
                        'type': seq_type.lower(),
                        'id': chain_id,
                        'name': user_name,
                        'sequence': sequence,
                        'modifications': mods_list if mods_list else None
                    }
                    all_sequences.append(seq_dict)

            elif seq_type.lower() == 'ligand':
                ligand_string = sequence.strip()

                # Check CCD lookup FIRST (before SMILES detection)
                # _is_smiles has false positives for CCD codes containing P,S,N,C,O,F,I
                looked_up_smiles = None
                is_known_ccd = False
                if ligand_string in self.ligand_lookup:
                    looked_up_smiles = self.ligand_lookup[ligand_string].get('smiles')
                    is_known_ccd = True
                elif ligand_string in self.ion_lookup:
                    looked_up_smiles = self.ion_lookup[ligand_string].get('smiles')
                    is_known_ccd = True

                if not is_known_ccd and not self._is_smiles(ligand_string):
                    errors.append(f"Unknown ligand/ion: '{ligand_string}'")

                for chain_id in chain_ids:
                    if looked_up_smiles:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'smiles': looked_up_smiles
                        }
                    elif is_known_ccd:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'ccd': ligand_string
                        }
                    else:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'smiles': ligand_string
                        }
                    all_sequences.append(seq_dict)

            else:
                errors.append(f"Unsupported sequence type: {seq_type}")

        if not all_sequences:
            errors.append("No valid sequences found")
            return None, errors

        remapped_pocket_binder = name_to_chain_mapping.get(pocket_binder, pocket_binder) if pocket_binder else pocket_binder
        remapped_contacts = self._remap_contact_chains(pocket_contacts, name_to_chain_mapping)

        pocket, pocket_errors = self._parse_pocket(remapped_pocket_binder, remapped_contacts)
        errors.extend(pocket_errors)

        remapped_bonds = self._remap_bond_chains(covalent_bonds_str, name_to_chain_mapping)
        covalent_bonds, bond_errors = self._parse_covalent_bonds(remapped_bonds)
        errors.extend(bond_errors)

        has_modifications = any(seq.get('modifications') for seq in all_sequences)

        job = {
            'name': jobname,
            'sequences': all_sequences,
            'pocket': pocket,
            'covalent_bonds': covalent_bonds,
            'has_modifications': has_modifications,
            'has_pocket': pocket is not None,
            'has_covalent': len(covalent_bonds) > 0
        }

        return job, errors

    def process_csv(self, csv_path: str) -> Tuple[List[Dict], pd.DataFrame]:
        """Process CSV file and return jobs list and summary DataFrame"""
        df = pd.read_csv(csv_path)

        jobs = []
        summary_rows = []

        for idx, row in df.iterrows():
            job, errors = self._process_job(row)

            if job:
                jobs.append(job)
                summary_rows.append({
                    'jobname': job['name'],
                    'sequences': len(job['sequences']),
                    'has_modifications': job['has_modifications'],
                    'has_pocket': job['has_pocket'],
                    'has_covalent': job['has_covalent'],
                    'status': 'ERROR' if errors else 'OK',
                    'errors': '; '.join(errors) if errors else ''
                })
            else:
                summary_rows.append({
                    'jobname': f"Row {idx+1}",
                    'sequences': 0,
                    'has_modifications': False,
                    'has_pocket': False,
                    'has_covalent': False,
                    'status': 'FAILED',
                    'errors': '; '.join(errors)
                })

        summary_df = pd.DataFrame(summary_rows)
        return jobs, summary_df

print(f"{chr(0x1f4e7)} Initializing IntelliFold CSV Processor...")
intellifold_processor = IntelliFoldJobProcessor()

print(f"{chr(0x2705)} Using embedded reference data: 79 entries")
print(f"{chr(0x2705)} Processor ready")
print(f"{chr(0x1f4cb)} Reference data includes:")
print(f"   - 15 PTM types")
print(f"   - 24 ligand types")
print(f"   - 11 ion types")
print(f"   - 10 glycan types")
print(f"   - 8 DNA modification types")
print(f"   - 10 RNA modification types")
print(f"\n{chr(0x1f4a1)} Using embedded reference data (common PTMs, ligands, ions, glycans, DNA/RNA mods)")
print("   To use custom reference: upload file in Cell 3")
print(f"\n{chr(0x1f4cb)} Note: Chain IDs are assigned as A, B, C, D... sequentially")
print("   Sequence identity is preserved in job names")

# ============================================================
# PROCESS UPLOADED CSV
# ============================================================
import pandas as pd

if 'global_settings' not in globals() or 'csv_filename' not in global_settings:
    print("Error: Please run Cell 2 (Upload CSV) first")
    raise ValueError("No CSV uploaded")

csv_filename = global_settings['csv_filename']

# Initialize processor
custom_ref_file = None
try:
    intellifold_processor = IntelliFoldJobProcessor(custom_ref_file)
except Exception as e:
    print(f"{chr(0x274c)} Failed to initialize processor: {e}")
    raise

print(f"\n{chr(0x1f504)} Processing CSV...")
try:
    jobs, summary_df = intellifold_processor.process_csv(csv_filename)
except Exception as e:
    print(f"{chr(0x274c)} CSV processing failed: {e}")
    raise

print("\n" + "=" * 60)
print(f"{chr(0x1f4cb)} JOB SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

if jobs:
    print("\n" + "=" * 60)
    print("YAML PREVIEW (first job)")
    print("=" * 60)
    preview_yaml = intellifold_processor._generate_yaml(jobs[0])
    print(preview_yaml)

error_jobs = summary_df[summary_df['status'] == 'ERROR']
if len(error_jobs) > 0:
    print(f"\n{chr(0x26a0)}  {len(error_jobs)} jobs have errors:")
    for _, row in error_jobs.iterrows():
        print(f"  - {row['jobname']}: {row['errors']}")

    proceed = input("\nProceed with valid jobs only? (yes/no): ").strip().lower()
    if proceed not in ['yes', 'y']:
        raise ValueError("Processing cancelled by user")

global_settings.update({
    'batch_jobs': jobs,
    'processor': intellifold_processor,
})

print("\n" + "=" * 60)
print(f"{chr(0x2705)} READY TO PROCESS {len(jobs)} JOBS")
print("=" * 60)
if global_settings.get('msa_folder'):
    print(f"Pre-computed MSAs: {global_settings['msa_folder']}")

print(f"\n{chr(0x1f4cc)} Next: Configure settings (Cell 5), then run (Cell 6)")

In [ ]:
#@title Cell 5: Configure IntelliFold Prediction Parameters

print("=" * 60)
print(f"{chr(0x1f4e7)} INTELLIFOLD PREDICTION PARAMETERS")
print("=" * 60)

# Model Selection
model = "v2" #@param ["v2", "v2-flash"]
#@markdown - **Model variant**: **v2** is the full model (highest accuracy, outperforms AlphaFold3 on Foldbench). **v2-flash** is faster with slightly lower accuracy. v2-flash is the CLI default, but v2 is recommended for best results.

# Core Model Parameters
recycling_iters = 10 #@param {type:"integer"}
#@markdown - **Recycling iterations**: Number of times the model refines its prediction. More iterations improve accuracy but increase runtime. IntelliFold default is 10 (vs Boltz-2's 6). **Time**: ~Linear scaling (10 iters ~ 1.7x time of 6 iters). **VRAM**: Minimal impact.

sampling_steps = 200 #@param {type:"integer"}
#@markdown - **Diffusion sampling steps**: Quality vs speed tradeoff for structure generation. More steps = smoother, higher quality structures. **Time**: Linear scaling (50 steps = 4x faster than 200). **VRAM**: +10-15% for intermediate diffusion states.

num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - **Number of structure samples**: Independent structure predictions per input. More samples increase diversity and reliability. **Time**: Linear scaling (5 samples = 5x base time). **VRAM**: Memory allocated per sample based on molecular size.

# Computation Settings
precision = "bf16" #@param ["bf16", "fp32", "fp16"]
#@markdown - **Computation precision**: bf16 (bfloat16) is recommended for best speed/accuracy balance. fp32 for maximum precision (slower). fp16 for maximum speed (may reduce accuracy). **Time**: bf16 ~ 2x faster than fp32. **VRAM**: bf16 ~ 50% less than fp32.

seed = "42" #@param {type:"string"}
#@markdown - **Random seed(s)**: Single int (e.g., "42") or comma-separated for multiple runs (e.g., "42,43,44,45,46"). Controls reproducibility. **Time**: Multiple seeds run sequentially. **VRAM**: No impact per seed.

num_workers = 4 #@param {type:"integer"}
#@markdown - **Dataloader workers**: Number of parallel CPU threads for data loading. More workers can speed up preprocessing but use more RAM. **Time**: Faster data loading with more workers. **VRAM**: No GPU impact, but increases CPU RAM usage.

cache_dir = "~/.intellifold" #@param {type:"string"}
#@markdown - **Cache directory**: Where IntelliFold stores model weights and reference data. Default is ~/.intellifold. **Time**: No impact. **VRAM**: No impact.

# MSA Settings
#@markdown > **Note**: These MSA settings only apply when `msa_folder` is empty (Cell 4).
#@markdown > When pre-computed MSAs are provided, online MSA search is skipped entirely.

use_msa_server = True #@param {type:"boolean"}
#@markdown - **Use MSA server**: Automatically generate MSAs using MMseqs2 ColabFold server. If disabled, you must provide pre-computed MSA files. **Time**: Adds 30-120 seconds per protein sequence. **VRAM**: No impact.

msa_server_url = "https://api.colabfold.com" #@param {type:"string"}
#@markdown - **MSA server URL**: Server endpoint for MSA generation. Default is ColabFold's public API. **Time**: No impact. **VRAM**: No impact.

msa_pairing_strategy = "greedy" #@param ["greedy", "complete"]
#@markdown - **MSA pairing strategy**: How to pair MSAs for multimers. 'greedy' pairs based on species taxonomy. 'complete' generates all possible pairings (slower but more thorough). **Time**: 'complete' can be significantly slower for many sequences. **VRAM**: Minimal impact.

no_pairing = False #@param {type:"boolean"}
#@markdown - **Disable MSA pairing**: When enabled, skips MSA pairing for multimers. Useful when chains are from different species with no overlapping homologs. **Time**: Slightly faster. **VRAM**: No impact.

# Output Settings
output_format = "mmcif" #@param ["mmcif", "pdb"]
#@markdown - **Structure file format**: mmCIF supports more metadata and modern features. PDB is more widely compatible. Both contain same structural information. **Time**: No impact. **VRAM**: No impact.

skip_existing = False  #@param {type:"boolean"}
#@markdown - **Skip existing**: If True, skip jobs that already have output in their base directory. Useful for resuming interrupted batches.

#@markdown ---
#@markdown ### Interface Analysis (ipSAE_batch)
#@markdown Scores interface quality (ipSAE, ipTM, pDockQ) and generates visualizations for each prediction.

enable_ipsae = True #@param {type:"boolean"}
#@markdown - **Enable**: Run ipSAE interface analysis on each completed job.

ipsae_png = True #@param {type:"boolean"}
#@markdown - **PNG plots**: Matrix + ribbon visualizations per model (~10-30s/job).

ipsae_pdf = False #@param {type:"boolean"}
#@markdown - **PDF report**: Side-by-side model comparison per job (~20-60s/job).

ipsae_per_residue = False #@param {type:"boolean"}
#@markdown - **Per-residue CSV**: Detailed per-residue interface scores.

ipsae_per_contact = False #@param {type:"boolean"}
#@markdown - **Per-contact CSV**: Per-contact-pair scores (most detailed).

ipsae_pae_cutoff = 10.0 #@param {type:"number"}
#@markdown - **PAE cutoff**: PAE threshold for interface scoring (default: 10.0).

ipsae_dist_cutoff = 10.0 #@param {type:"number"}
#@markdown - **Distance cutoff**: CB-CB distance threshold in Angstroms (default: 10.0).

ipsae_select_residues = "" #@param {type:"string"}
#@markdown - **Select residues**: Focus on specific residues, e.g. `A:100-105,B:203`. Empty = all interfaces.

ipsae_batch_analysis = True #@param {type:"boolean"}
#@markdown - **Final batch comparison**: Combined CSV + interactive HTML across all jobs.

# Advanced Settings (Optional)
print(f"\n{chr(0x2699)}  Advanced settings configured with defaults")
print("   Most users should keep these as-is")

# Check if global_settings exists
if 'global_settings' not in globals():
    print(f"\n{chr(0x274c)} Error: Please run Cell 3 (CSV Upload) first")
    raise ValueError("No batch jobs configured")

# Store all settings
global_settings['config'] = {
    'model': model,
    'recycling_iters': recycling_iters,
    'sampling_steps': sampling_steps,
    'num_diffusion_samples': num_diffusion_samples,
    'precision': precision,
    'seed': seed,
    'num_workers': num_workers,
    'cache_dir': cache_dir,
    'use_msa_server': use_msa_server,
    'msa_server_url': msa_server_url,
    'msa_pairing_strategy': msa_pairing_strategy,
    'no_pairing': no_pairing,
    'output_format': output_format,
    'skip_existing': skip_existing,
    'enable_ipsae': enable_ipsae,
    'ipsae_png': ipsae_png,
    'ipsae_pdf': ipsae_pdf,
    'ipsae_per_residue': ipsae_per_residue,
    'ipsae_per_contact': ipsae_per_contact,
    'ipsae_pae_cutoff': ipsae_pae_cutoff,
    'ipsae_dist_cutoff': ipsae_dist_cutoff,
    'ipsae_select_residues': ipsae_select_residues,
    'ipsae_batch_analysis': ipsae_batch_analysis,
}

# Display summary
print("\n" + "=" * 60)
print(f"{chr(0x1f4cb)} CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Model Settings:")
print(f"  - Model: IntelliFold {model}")
print(f"  - Recycling iterations: {recycling_iters}")
print(f"  - Sampling steps: {sampling_steps}")
print(f"  - Diffusion samples: {num_diffusion_samples}")
print(f"  - Precision: {precision}")
print(f"  - Random seed(s): {seed}")
print(f"\nComputation:")
print(f"  - Workers: {num_workers}")
print(f"  - Cache: {cache_dir}")
print(f"\nMSA Generation:")
print(f"  - Use MSA server: {use_msa_server}")
print(f"  - Server URL: {msa_server_url}")
print(f"  - Pairing strategy: {msa_pairing_strategy}")
print(f"  - No pairing: {no_pairing}")
print(f"\nOutput:")
print(f"  - Format: {output_format}")
print(f"  - Skip existing: {skip_existing}")


print(f"\nInterface Analysis (ipSAE):")
print(f"  - Enabled: {enable_ipsae}")
if enable_ipsae:
    print(f"  - PNG: {ipsae_png} | PDF: {ipsae_pdf}")
    print(f"  - Per-residue: {ipsae_per_residue} | Per-contact: {ipsae_per_contact}")
    print(f"  - Cutoffs: PAE={ipsae_pae_cutoff}, dist={ipsae_dist_cutoff}")
    if ipsae_select_residues:
        print(f"  - Selected residues: {ipsae_select_residues}")
    print(f"  - Final batch comparison: {ipsae_batch_analysis}")

print(f"\n{chr(0x1f4c1)} Next: Run Cell 6 to start batch predictions")

In [ ]:
#@title Cell 6: Run Batch Predictions with Calibration-Based Parallel GPU Scheduling
#@markdown **Runs job 1 alone to calibrate VRAM usage, then launches remaining jobs in parallel when GPU memory allows.**
#@markdown Results are automatically uploaded to Google Drive after each job completes.
%%time

import subprocess
import os
import zipfile
import time
import gc
import threading
import queue
import re
from datetime import datetime
import json

# Fast LayerNorm: 30-50% speedup via fused CUDA kernels
os.environ["LAYERNORM_TYPE"] = "fast_layernorm"


def get_unique_job_dir(job_name):
    """Return a unique job directory, appending _2, _3, etc. if exists."""
    if not os.path.exists(job_name):
        return job_name
    n = 2
    while os.path.exists(f"{job_name}_{n}"):
        n += 1
    return f"{job_name}_{n}"

def has_existing_output(job_name, search_dir="/content"):
    """Return True if job_name/ contains any .cif or .pdb structure files."""
    job_dir = os.path.join(search_dir, job_name)
    if not os.path.isdir(job_dir):
        return False
    for root, _, files in os.walk(job_dir):
        for f in files:
            if f.endswith(('.cif', '.pdb')):
                return True
    return False

# ============================================================
# VERIFY SETUP
# ============================================================
if 'global_settings' not in globals():
    print(f"{chr(0x274c)} Error: Please run the previous configuration cells first")
    raise ValueError("Configuration not found")

if not global_settings.get('batch_jobs'):
    print(f"{chr(0x274c)} Error: No batch jobs configured")
    print("   Please run Cell 3 to upload and process your CSV")
    raise ValueError("No jobs to process")

config = global_settings['config']
jobs = global_settings['batch_jobs']
processor = global_settings['processor']
drive = global_settings.get('drive')
gdrive_folder_name = global_settings.get('gdrive_folder_name', 'IntelliFold_Results')


# Check ipSAE_batch availability
ipsae_available = False
try:
    result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True, timeout=10)
    ipsae_available = (result.returncode == 0)
except Exception:
    pass
if config.get('enable_ipsae') and not ipsae_available:
    print("WARNING: ipSAE_batch not installed. Interface analysis will be skipped.")
# ============================================================
# PARALLEL EXECUTION SETTINGS
# ============================================================
enable_parallel = True #@param {type:"boolean"}
#@markdown - **Enable parallel execution**: Launch multiple jobs simultaneously when VRAM allows
#@markdown > **Note**: Jobs are automatically sorted largest-first for optimal calibration. The largest job calibrates VRAM, giving the most accurate per-token rate for parallel scheduling.

vram_limit = 0.85 #@param {type:"slider", min:0.6, max:0.95, step:0.05}
#@markdown - **VRAM limit**: Max fraction of GPU memory to use (0.85 = 85%). Jobs won't launch if this would be exceeded.

# ============================================================
# GPU VERIFICATION
# ============================================================
print("=" * 60)
print(f"{chr(0x1f50d)} GPU VERIFICATION")
print("=" * 60)
gpu_available = False
total_gpu_vram = 0
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        total_gpu_vram = gpu_memory_gb
        print(f"{chr(0x2705)} GPU: {gpu_name}")
        print(f"   Total Memory: {gpu_memory_gb:.1f} GB")
        print(f"   VRAM limit ({vram_limit*100:.0f}%): {gpu_memory_gb * vram_limit:.1f} GB")

        test_tensor = torch.zeros(1).cuda()
        del test_tensor
        torch.cuda.synchronize()
        gpu_available = True

        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            if result.returncode == 0:
                initial_mem = float(result.stdout.strip()) / 1024
                print(f"   Current Usage: {initial_mem:.2f} GB")
                print(f"   {chr(0x2705)} GPU monitoring ready")
            else:
                print(f"   {chr(0x26a0)}  nvidia-smi not available, parallel mode disabled")
                enable_parallel = False
        except Exception as e:
            print(f"   {chr(0x26a0)}  nvidia-smi test failed: {e}")
            enable_parallel = False
    else:
        print(f"{chr(0x26a0)}  WARNING: No GPU detected")
        print("   Predictions will be very slow on CPU")
        enable_parallel = False
except Exception as e:
    print(f"{chr(0x26a0)}  GPU test failed: {e}")
    gpu_available = False
    enable_parallel = False

# Precision check
if config.get('precision') == 'fp16':
    print("\n" + f"{chr(0x26a0)}  " * 20)
    print(f"{chr(0x26a0)}  WARNING: Using FP16 precision")
    print(f"{chr(0x26a0)}  FP16 can cause numerical instability (NaN/Inf values)")
    print(f"{chr(0x26a0)}  RECOMMENDATION: Change Cell 4 precision to 'bf16' for better stability")
    print(f"{chr(0x26a0)}  " * 20)
elif config.get('precision') == 'bf16':
    print(f"\n{chr(0x2705)} Using BF16 precision - good balance of speed and stability")
else:
    print(f"\n{chr(0x2705)} Using {config.get('precision')} precision")

# ============================================================
# GPU MONITOR CLASS
# ============================================================
class GPUMonitor:
    """Monitor GPU memory usage using nvidia-smi"""

    def __init__(self):
        self.monitoring = False
        self.peak_memory = 0
        self.current_memory = 0
        self.monitor_thread = None
        self._lock = threading.Lock()
        self.gpu_available = self._test_nvidia_smi()
        self._pid_registry = {}  # shell_pid -> {name, peak_gb}

    def _test_nvidia_smi(self):
        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            return result.returncode == 0
        except:
            return False

    def _get_gpu_memory(self):
        if not self.gpu_available:
            return 0
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if result.returncode == 0:
                return float(result.stdout.strip()) / 1024  # MB to GB
        except:
            pass
        return 0

    def _monitor_loop(self):
        while self.monitoring:
            mem = self._get_gpu_memory()
            # Per-PID VRAM tracking
            job_mem = {}
            with self._lock:
                need_pid = bool(self._pid_registry)
                shells = set(self._pid_registry.keys()) if need_pid else set()
            if need_pid:
                pid_mem = self._get_pid_mem()
                for gpu_pid, gb in pid_mem.items():
                    shell = self._find_job_for_gpu_pid(gpu_pid, shells)
                    if shell is not None:
                        job_mem[shell] = job_mem.get(shell, 0) + gb
            with self._lock:
                self.current_memory = mem
                if mem > self.peak_memory:
                    self.peak_memory = mem
                for spid, cur_gb in job_mem.items():
                    if spid in self._pid_registry:
                        if cur_gb > self._pid_registry[spid]['peak_gb']:
                            self._pid_registry[spid]['peak_gb'] = cur_gb
            time.sleep(0.5)

    def start(self):
        if not self.gpu_available:
            return
        with self._lock:
            self.monitoring = True
            self.peak_memory = 0
            self.current_memory = 0
        self.monitor_thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.monitor_thread.start()

    def stop(self):
        self.monitoring = False
        if self.monitor_thread:
            self.monitor_thread.join(timeout=2)
        return self.peak_memory

    def get_current(self):
        with self._lock:
            return self.current_memory

    def get_peak(self):
        with self._lock:
            return self.peak_memory

    def reset_peak(self):
        with self._lock:
            self.peak_memory = self.current_memory

    def _get_pid_mem(self):
        """Get per-PID GPU memory {pid: GB}."""
        try:
            r = subprocess.run(
                ['nvidia-smi', '--query-compute-apps=pid,used_gpu_memory', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if r.returncode == 0 and r.stdout.strip():
                result = {}
                for ln in r.stdout.strip().split('\n'):
                    parts = ln.strip().split(',')
                    if len(parts) >= 2:
                        try:
                            result[int(parts[0].strip())] = float(parts[1].strip()) / 1024
                        except ValueError:
                            pass
                return result
        except:
            pass
        return {}

    def _find_job_for_gpu_pid(self, gpu_pid, registered_shells):
        """Walk process tree up to find which registered job owns this GPU PID."""
        current = gpu_pid
        visited = set()
        while current > 1 and current not in visited:
            visited.add(current)
            if current in registered_shells:
                return current
            try:
                with open(f'/proc/{current}/status') as f:
                    for line in f:
                        if line.startswith('PPid:'):
                            current = int(line.split(':')[1].strip())
                            break
                    else:
                        break
            except (FileNotFoundError, PermissionError, ValueError):
                break
        return None

    def register_pid(self, shell_pid, job_name):
        """Register a job process PID for per-job VRAM tracking."""
        with self._lock:
            self._pid_registry[shell_pid] = {'name': job_name, 'peak_gb': 0.0}

    def unregister_pid(self, shell_pid):
        """Unregister and return peak VRAM (GB) for a job."""
        with self._lock:
            entry = self._pid_registry.pop(shell_pid, None)
            return entry['peak_gb'] if entry else 0.0

    def get_pid_peak(self, shell_pid):
        """Get peak VRAM (GB) for a specific registered job."""
        with self._lock:
            entry = self._pid_registry.get(shell_pid)
            return entry['peak_gb'] if entry else 0.0


gpu_monitor = GPUMonitor()

def clear_gpu_cache():
    if not gpu_available:
        return
    try:
        import torch
        torch.cuda.empty_cache()
        gc.collect()
    except:
        pass

# ============================================================
# GOOGLE DRIVE HELPERS
# ============================================================
def get_or_create_folder(drive, folder_name, parent_id='root'):
    if not drive:
        return None
    try:
        file_list = drive.ListFile({
            'q': f"'{parent_id}' in parents and title='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        }).GetList()
        if file_list:
            print(f"{chr(0x2705)} Found existing folder: {folder_name}")
            return file_list[0]['id']
        else:
            folder = drive.CreateFile({
                'title': folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [{'id': parent_id}]
            })
            folder.Upload()
            print(f"{chr(0x2705)} Created new folder: {folder_name}")
            return folder['id']
    except Exception as e:
        print(f"{chr(0x274c)} Error with folder '{folder_name}': {e}")
        return None

def upload_to_gdrive(drive, file_path, folder_id, job_name):
    if not drive or not os.path.exists(file_path):
        return None
    try:
        uploaded_file = drive.CreateFile({
            'title': os.path.basename(file_path),
            'parents': [{'id': folder_id}]
        })
        uploaded_file.SetContentFile(file_path)
        uploaded_file.Upload()
        file_url = f"https://drive.google.com/file/d/{uploaded_file['id']}/view"
        print(f"  {chr(0x2601)}  Uploaded to Google Drive: {file_url}")
        return file_url
    except Exception as e:
        print(f"  {chr(0x26a0)}  Upload failed: {e}")
        return None

# ============================================================
# UTILITY FUNCTIONS
# ============================================================
def read_error_files(job_dir, job_name):
    error_info = []
    error_dirs = [
        os.path.join(job_dir, "errors"),
        os.path.join(job_dir, job_name, "errors"),
    ]
    for error_dir in error_dirs:
        if os.path.exists(error_dir):
            try:
                for f in os.listdir(error_dir):
                    if f.endswith(('.txt', '.log', '.json')):
                        try:
                            with open(os.path.join(error_dir, f), 'r') as fh:
                                error_info.append({'file': f, 'content': fh.read()[:500]})
                        except:
                            pass
            except:
                pass
    return error_info

def find_predictions_directory(job_dir, job_name):
    possible_paths = [
        job_dir,
        os.path.join(job_dir, job_name),
        os.path.join(job_dir, "predictions"),
        os.path.join(job_dir, job_name, "predictions"),
        os.path.join(job_dir, "predictions", job_name),
    ]
    for path in possible_paths:
        if os.path.exists(path):
            try:
                for root, dirs, files in os.walk(path):
                    for file in files:
                        if file.endswith(('.cif', '.pdb', '.mmcif')):
                            return root
            except:
                continue
    return None

def parse_intellifold_progress(line):
    keywords = ['progress', 'step', 'recycling', 'diffusion', 'sample',
                'completed', 'saving', 'msa', 'model', 'generating',
                'loading', 'processing', 'predicting']
    line_lower = line.lower()
    return any(kw in line_lower for kw in keywords)

def count_job_tokens(job, processor):
    total_tokens = 0
    for seq in job.get('sequences', []):
        seq_type = seq.get('type', '').lower()
        if seq_type in ['protein', 'dna', 'rna']:
            total_tokens += len(seq.get('sequence', ''))
            modifications = seq.get('modifications')
            if modifications:
                for mod in modifications:
                    mod_code = mod.get('ptmType') or mod.get('modificationType', '')
                    if seq_type == 'protein' and mod_code in processor.ptm_lookup:
                        total_tokens += processor.ptm_lookup[mod_code].get('atom_count', 0)
                    elif seq_type == 'dna' and mod_code in processor.dna_mod_lookup:
                        total_tokens += processor.dna_mod_lookup[mod_code].get('atom_count', 0)
                    elif seq_type == 'rna' and mod_code in processor.rna_mod_lookup:
                        total_tokens += processor.rna_mod_lookup[mod_code].get('atom_count', 0)
        elif seq_type == 'ligand':
            ccd_code = seq.get('ccd')
            smiles = seq.get('smiles')
            if ccd_code:
                if ccd_code in processor.ligand_lookup:
                    total_tokens += processor.ligand_lookup[ccd_code].get('atom_count', 20)
                elif ccd_code in processor.ion_lookup:
                    total_tokens += processor.ion_lookup[ccd_code].get('atom_count', 1)
                elif ccd_code in processor.glycan_lookup:
                    total_tokens += processor.glycan_lookup[ccd_code].get('atom_count', 15)
                else:
                    total_tokens += 20
            elif smiles:
                total_tokens += 30
    return max(total_tokens, 1)

# ============================================================
# NON-BLOCKING PROCESS OUTPUT READER
# ============================================================

# ============================================================
# MSA FOLDER LOOKUP HELPER
# ============================================================
def find_precomputed_msas(job, msa_folder):
    """Look up pre-computed MSA files for ALL protein chains in a job.

    All-or-nothing: returns MSA paths only if EVERY protein chain has a
    pre-computed MSA file. If any chain is missing, returns empty dict
    and the job falls back to online MSA search entirely.
    """
    if not msa_folder or not os.path.isdir(msa_folder):
        return {}

    job_name = job['name']
    found = {}
    missing = []

    seen = set()
    protein_chains = []
    for seq in job.get('sequences', []):
        if seq.get('type', '').lower() != 'protein':
            continue
        chain_name = seq.get('name', '')
        if not chain_name or chain_name in seen:
            continue
        seen.add(chain_name)
        protein_chains.append(chain_name)

    if not protein_chains:
        return {}

    available_files = {}
    for fname in os.listdir(msa_folder):
        if fname.endswith('_paired.a3m'):
            normalized = fname.lower().replace('-', '_')
            available_files[normalized] = fname

    for chain_name in protein_chains:
        expected = f"{job_name}_{chain_name}_paired.a3m"
        exact_path = os.path.join(msa_folder, expected)
        if os.path.isfile(exact_path):
            found[chain_name] = exact_path
        else:
            norm_key = expected.lower().replace('-', '_')
            if norm_key in available_files:
                found[chain_name] = os.path.join(msa_folder, available_files[norm_key])
            else:
                missing.append(chain_name)

    if missing:
        print(f"   Pre-computed MSAs missing for: {missing} -- using online MSA for all chains")
        return {}

    print(f"   Pre-computed MSAs found for all {len(found)} chain(s): {list(found.keys())}")
    return found

def _reader_thread(pipe, output_queue):
    """Read lines from a pipe into a thread-safe queue."""
    try:
        for line in iter(pipe.readline, ''):
            output_queue.put(line.rstrip('\n'))
    except:
        pass
    finally:
        try:
            pipe.close()
        except:
            pass

# ============================================================
# BUILD INTELLIFOLD COMMAND
# ============================================================
def build_intellifold_cmd(yaml_file, job_dir, current_config):
    cmd_parts = [
        "intellifold", "predict", yaml_file,
        "--out_dir", job_dir,
        "--model", current_config['model'],
        "--recycling_iters", str(current_config['recycling_iters']),
        "--sampling_steps", str(current_config['sampling_steps']),
        "--num_diffusion_samples", str(current_config['num_diffusion_samples']),
        "--precision", current_config['precision'],
        "--seed", current_config['seed'],
        "--output_format", current_config['output_format'],
        "--cache", current_config['cache_dir'],
        "--num_workers", str(current_config['num_workers'])
    ]
    if current_config['use_msa_server'] and not global_settings.get('msa_folder'):
        cmd_parts.extend([
            "--use_msa_server",
            "--msa_server_url", current_config['msa_server_url'],
            "--msa_pairing_strategy", current_config['msa_pairing_strategy']
        ])
    if current_config.get('no_pairing', False):
        cmd_parts.append("--no_pairing")
    return " ".join(cmd_parts)


# ============================================================
# ipSAE INTERFACE ANALYSIS
# ============================================================
def run_ipsae_analysis(job_dir, job_name):
    """Run ipSAE per-job analysis. Returns True on success, False on failure."""
    if not ipsae_available or not config.get('enable_ipsae'):
        return False

    ipsae_output = os.path.join(job_dir, "ipsae_analysis")
    os.makedirs(ipsae_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", job_dir,
        "--backend", "intellifold",
        "--output_dir", ipsae_output,
        "--pae_cutoff", str(config.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(config.get('ipsae_dist_cutoff', 10.0)),
        "--workers", "1",
        "--cores", "1",
    ]
    if config.get('ipsae_png'):
        cmd_parts.append("--png")
    if config.get('ipsae_pdf'):
        cmd_parts.append("--pdf")
    if config.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if config.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if config.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", config['ipsae_select_residues']])

    print(f"   Running ipSAE analysis...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            n_files = sum(1 for f in os.listdir(ipsae_output) if not f.startswith('.'))
            print(f"   ipSAE complete: {n_files} output files")
            return True
        else:
            print(f"   ipSAE failed (rc={result.returncode}): {result.stderr[:200]}")
            return False
    except subprocess.TimeoutExpired:
        print(f"   ipSAE timed out (300s)")
        return False
    except Exception as e:
        print(f"   ipSAE error: {e}")
        return False

# ============================================================
# PROCESS COMPLETED JOB (zip + upload + collect results)
# ============================================================
def process_completed_job(job_name, job_dir, yaml_file, job_time, peak_gpu, drive, gdrive_folder_id):
    """Find outputs, zip, upload to GDrive. Returns result dict."""
    predictions_dir = find_predictions_directory(job_dir, job_name)

    if not predictions_dir:
        print(f"\n{chr(0x274c)} No structure files (.cif/.pdb/.mmcif) found for {job_name}")
        error_files = read_error_files(job_dir, job_name)
        if error_files:
            for err_info in error_files[:2]:
                print(f"   Error file {err_info['file']}: {err_info['content'][:200]}")
        # Show directory structure
        print(f"   {chr(0x1f4c2)} Directory structure of {job_dir}:")
        for root, dirs, files in os.walk(job_dir):
            level = root.replace(job_dir, '').count(os.sep)
            indent = '   ' * (level + 1)
            print(f"{indent}{chr(0x1f4c1)} {os.path.basename(root)}/")
            for file in files[:10]:
                print(f"{indent}  {chr(0x1f4c4)} {file}")
        return {
            'success': False, 'name': job_name,
            'error': 'No structure files generated',
            'time': job_time, 'gpu_peak': peak_gpu
        }

    structure_files = [f for f in os.listdir(predictions_dir) if f.endswith(('.cif', '.pdb', '.mmcif'))]

    if not structure_files:
        print(f"\n{chr(0x274c)} No structure files in predictions directory for {job_name}")
        return {
            'success': False, 'name': job_name,
            'error': 'No structure files in predictions directory',
            'time': job_time, 'gpu_peak': peak_gpu
        }

    print(f"   {chr(0x2705)} {len(structure_files)} structure files generated")
    for f in sorted(structure_files)[:3]:
        print(f"      {chr(0x1f4c4)} {f}")
    if len(structure_files) > 3:
        print(f"      ... and {len(structure_files) - 3} more")

    # --- ipSAE interface analysis (before zipping) ---
    ipsae_ran = False
    try:
        ipsae_ran = run_ipsae_analysis(job_dir, job_name)
    except Exception as e:
        print(f"   ipSAE error (non-fatal): {e}")

    # Create ZIP
    zip_filename = f"{job_name}_results.zip"
    zip_path = os.path.join(job_dir, zip_filename)
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(predictions_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, job_dir)
                    zipf.write(file_path, arcname)
            zipf.write(yaml_file, os.path.basename(yaml_file))
            # Add environment log for traceability
            if os.path.isfile("/content/environment.txt"):
                zipf.write("/content/environment.txt", "environment.txt")
            # Add ipSAE analysis files
            ipsae_dir = os.path.join(job_dir, "ipsae_analysis")
            if os.path.isdir(ipsae_dir):
                for root, dirs, files_list in os.walk(ipsae_dir):
                    for f in files_list:
                        fp = os.path.join(root, f)
                        arcname = os.path.join("ipsae_analysis", os.path.relpath(fp, ipsae_dir))
                        zipf.write(fp, arcname)
        print(f"   {chr(0x1f4e6)} Created: {zip_filename}")
    except Exception as e:
        print(f"   {chr(0x26a0)}  Failed to create ZIP: {e}")

    # Upload to GDrive
    file_url = None
    if drive and gdrive_folder_id:
        file_url = upload_to_gdrive(drive, zip_path, gdrive_folder_id, job_name)
    else:
        print(f"   {chr(0x1f4c1)} Saved locally: {zip_path}")

    return {
        'success': True, 'name': job_name,
        'structures': len(structure_files),
        'time': job_time, 'url': file_url,
        'gpu_peak': peak_gpu,
        'ipsae_ran': ipsae_ran,
    }

# ============================================================
# SEQUENTIAL RUNNER (proven, from GPU_FIXED)
# ============================================================
def run_single_prediction_sequential(job, job_idx, total_jobs, retry_with_safer=False):
    """Run a single IntelliFold prediction sequentially with GPU monitoring."""
    job_start = time.time()

    print("\n" + "=" * 60)
    print(f"{chr(0x1f680)} Job {job_idx}/{total_jobs}: {job['name']}")
    if retry_with_safer:
        print(f"   {chr(0x1f504)} RETRY with safer precision settings")
    print("=" * 60)

    gpu_monitor.start()
    time.sleep(0.5)
    initial_mem = gpu_monitor.get_current()
    print(f"{chr(0x1f3ae)} GPU Memory (start): {initial_mem:.2f} GB")

    job_name = job['name']
    if config.get('skip_existing', False) and has_existing_output(job_name):
        print(f"  \u23ed Skipping {job_name}: output already exists")
        gpu_monitor.stop()
        return {'name': job_name, 'skipped': True, 'success': True,
                'structures': 0, 'time': 0, 'gpu_peak': 0,
                'tokens': count_job_tokens(job, processor)}
    job_dir = get_unique_job_dir(job_name)
    os.makedirs(job_dir, exist_ok=True)

    yaml_content = processor._generate_yaml(job)

    # Inject pre-computed MSA paths if available
    msa_folder_val = global_settings.get('msa_folder', '')
    if msa_folder_val:
        precomp_msas = find_precomputed_msas(job, msa_folder_val)
        if precomp_msas:
            import yaml as _yaml
            yaml_data = _yaml.safe_load(yaml_content)
            id_to_name = {}
            for seq in job.get('sequences', []):
                if seq.get('type', '').lower() == 'protein' and seq.get('name'):
                    id_to_name[seq.get('id', '')] = seq['name']
            for entry in yaml_data.get('sequences', []):
                if 'protein' in entry:
                    protein = entry['protein']
                    chain_id = protein.get('id')
                    ids = chain_id if isinstance(chain_id, list) else [chain_id]
                    for cid in ids:
                        chain_name = id_to_name.get(cid)
                        if chain_name and chain_name in precomp_msas:
                            protein['msa'] = precomp_msas[chain_name]
                            break
            yaml_content = _yaml.dump(yaml_data, default_flow_style=False, sort_keys=False)
    yaml_file = os.path.join(job_dir, f"{job_name}.yaml")
    with open(yaml_file, 'w') as f:
        f.write(yaml_content)

    current_config = config.copy()
    if retry_with_safer and current_config.get('precision') == 'fp16':
        current_config['precision'] = 'bf16'
        print(f"   {chr(0x1f527)} Changed precision: fp16 -> bf16")

    cmd = build_intellifold_cmd(yaml_file, job_dir, current_config)
    print(f"{chr(0x1f4e7)} Command: {cmd}")
    print(f"\n{chr(0x2699)}  Running IntelliFold prediction...")

    prediction_failed = False
    error_messages = []

    # Phase tracking
    phase_times = {}
    current_phase = "initialization"
    phase_start = time.time()

    try:
        process = subprocess.Popen(
            cmd, shell=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )

        # Non-blocking reader
        output_q = queue.Queue()
        reader_t = threading.Thread(target=_reader_thread, args=(process.stdout, output_q), daemon=True)
        reader_t.start()

        print(f"\n{chr(0x1f4ca)} Progress:")
        while process.poll() is None:
            while not output_q.empty():
                try:
                    line = output_q.get_nowait()
                except queue.Empty:
                    break
                if not line:
                    continue

                # Track errors
                if 'error' in line.lower() and 'nan or inf' in line.lower():
                    error_messages.append(line)
                    prediction_failed = True
                elif 'skipping the target' in line.lower():
                    error_messages.append(line)
                    prediction_failed = True

                # Track phases
                ll = line.lower()
                if 'msa' in ll and current_phase != "msa":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "msa"; phase_start = time.time()
                elif 'recycling' in ll and current_phase != "recycling":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "recycling"; phase_start = time.time()
                elif 'diffusion' in ll and current_phase != "diffusion":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "diffusion"; phase_start = time.time()
                elif 'saving' in ll and current_phase != "saving":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "saving"; phase_start = time.time()

                # Print progress
                if parse_intellifold_progress(line):
                    cur_gpu = gpu_monitor.get_current()
                    if cur_gpu > 1.0:
                        print(f"   {line[:100]} [GPU: {cur_gpu:.1f} GB]")
                    else:
                        print(f"   {line[:100]}")
                elif 'error' in line.lower() or 'warning' in line.lower():
                    print(f"   {chr(0x26a0)}  {line[:100]}")

            time.sleep(0.5)

        # Drain remaining output
        while not output_q.empty():
            try:
                line = output_q.get_nowait()
                if line and (parse_intellifold_progress(line) or 'error' in line.lower()):
                    print(f"   {line[:100]}")
            except queue.Empty:
                break

        phase_times[current_phase] = time.time() - phase_start
        return_code = process.wait(timeout=60)
        peak_gpu_memory = gpu_monitor.stop()

        print(f"\n{chr(0x1f3ae)} GPU Memory: Peak={peak_gpu_memory:.2f} GB, Final={gpu_monitor.get_current():.2f} GB")

        if return_code != 0:
            print(f"\n{chr(0x274c)} Prediction failed (return code: {return_code})")
            error_files = read_error_files(job_dir, job_name)
            if error_files:
                for err_info in error_files[:2]:
                    print(f"   {err_info['file']}: {err_info['content'][:200]}")
            return {
                'success': False, 'name': job_name,
                'error': f"Process exited with code {return_code}",
                'error_messages': error_messages,
                'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory,
                'can_retry': prediction_failed and not retry_with_safer
            }

    except subprocess.TimeoutExpired:
        process.kill()
        peak_gpu_memory = gpu_monitor.stop()
        return {
            'success': False, 'name': job_name, 'error': "Timeout",
            'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory, 'can_retry': False
        }
    except Exception as e:
        peak_gpu_memory = gpu_monitor.stop()
        return {
            'success': False, 'name': job_name, 'error': str(e),
            'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory, 'can_retry': False
        }

    # Print timing
    if phase_times:
        total_phase_time = sum(phase_times.values())
        print(f"\n{chr(0x23f1)}  Timing:")
        for phase, duration in phase_times.items():
            if duration > 0.1:
                pct = (duration / total_phase_time) * 100 if total_phase_time > 0 else 0
                print(f"   - {phase.capitalize()}: {duration:.1f}s ({pct:.0f}%)")

    job_time = time.time() - job_start
    result = process_completed_job(job_name, job_dir, yaml_file, job_time, peak_gpu_memory, drive, gdrive_folder_id)
    result['can_retry'] = prediction_failed and not retry_with_safer
    if 'tokens' not in result:
        result['tokens'] = count_job_tokens(job, processor)
    print(f"\n{chr(0x23f1)}  Total: {job_time:.1f}s ({job_time/60:.1f} min)")
    return result

# ============================================================
# SETUP GOOGLE DRIVE
# ============================================================
gdrive_folder_id = None
if drive:
    print("\n" + "=" * 60)
    print(f"{chr(0x2601)}  SETTING UP GOOGLE DRIVE")
    print("=" * 60)
    gdrive_folder_id = get_or_create_folder(drive, gdrive_folder_name)
    if gdrive_folder_id:
        print(f"{chr(0x2705)} Results will be uploaded to: {gdrive_folder_name}")
    else:
        print(f"{chr(0x26a0)}  Could not setup Drive folder - results will be saved locally only")
        drive = None
else:
    print("\n" + "=" * 60)
    print(f"{chr(0x1f4c1)} NO GOOGLE DRIVE - LOCAL ONLY")
    print("=" * 60)
    print(f"{chr(0x26a0)}  Google Drive not configured in Cell 3")

# ============================================================
# PRINT BATCH INFO
# ============================================================
print("\n" + "=" * 60)
print(f"{chr(0x1f680)} STARTING BATCH PROCESSING: {len(jobs)} JOBS")
print("=" * 60)
print(f"Configuration:")
print(f"  - Recycling iters: {config['recycling_iters']}")
print(f"  - Sampling steps: {config['sampling_steps']}")
print(f"  - Diffusion samples: {config['num_diffusion_samples']}")
print(f"  - Precision: {config['precision']}")
print(f"  - Seeds: {config['seed']}")
print(f"  - MSA server: {config['use_msa_server']}")
print(f"  - Output format: {config['output_format']}")
print(f"  - Parallel mode: {enable_parallel}")
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Sort jobs largest-first for optimal calibration accuracy
jobs = sorted(jobs, key=lambda j: count_job_tokens(j, processor), reverse=True)
print(f"Jobs sorted by size (largest first):")
for _j in jobs:
    print(f"  {_j['name']}: {count_job_tokens(_j, processor)} tokens")

# ============================================================
# SINGLE JOB SHORTCUT
# ============================================================
if len(jobs) == 1:
    enable_parallel = False

# ============================================================
# MAIN EXECUTION
# ============================================================
batch_start_time = time.time()
completed_jobs = []
failed_jobs = []
retried_jobs = []

if not enable_parallel or not gpu_available:
    # --------------------------------------------------------
    # SEQUENTIAL MODE
    # --------------------------------------------------------
    if not enable_parallel:
        print(f"\n{chr(0x1f504)} Running in SEQUENTIAL mode (parallel disabled)")
    else:
        print(f"\n{chr(0x1f504)} Running in SEQUENTIAL mode (no GPU monitoring)")

    for job_idx, job in enumerate(jobs, 1):
        result = run_single_prediction_sequential(job, job_idx, len(jobs))

        if result['success']:
            completed_jobs.append(result)
        else:
            if result.get('can_retry') and 'nan or inf' in str(result.get('error_messages', [])).lower():
                print(f"\n{chr(0x1f504)} Retrying {job['name']} with safer precision...")
                retried_jobs.append(job['name'])
                clear_gpu_cache()
                time.sleep(2)
                result = run_single_prediction_sequential(job, job_idx, len(jobs), retry_with_safer=True)
                if result['success']:
                    completed_jobs.append(result)
                else:
                    failed_jobs.append(result)
            else:
                failed_jobs.append(result)

        clear_gpu_cache()
        time.sleep(1)

else:
    # --------------------------------------------------------
    # CALIBRATION-BASED PARALLEL MODE
    # --------------------------------------------------------
    print(f"\n" + "=" * 60)
    print(f"{chr(0x1f9ea)} CALIBRATION-BASED PARALLEL MODE")
    print("=" * 60)
    print(f"Phase 1: Run job 1 alone to measure actual VRAM usage")
    print(f"Phase 2: Use measurement to estimate remaining jobs")
    print(f"Phase 3: Launch jobs in parallel when VRAM allows")
    print("=" * 60)

    # --- PHASE 1: CALIBRATION (try jobs one by one until one succeeds) ---
    print(f"\n{chr(0x1f4cf)} PHASE 1: CALIBRATION")
    print(f"Running jobs one at a time until one succeeds to measure VRAM...")

    cal_result = None
    cal_peak_vram = 0
    cal_tokens = 0
    calibration_idx = -1  # index of the job that succeeded calibration

    skipped_in_cal = 0
    for cal_idx, cal_job in enumerate(jobs):
        cal_job_name = cal_job['name']

        # Skip already-completed jobs -- they give no VRAM data
        if config.get('skip_existing', False) and has_existing_output(cal_job_name):
            print(f"  \u23ed Skipping {cal_job_name} (already done)")
            completed_jobs.append({'name': cal_job_name, 'skipped': True, 'success': True,
                                   'structures': 0, 'time': 0, 'gpu_peak': 0,
                                   'tokens': count_job_tokens(cal_job, processor)})
            skipped_in_cal += 1
            continue

        cal_tokens = count_job_tokens(cal_job, processor)
        print(f"\n   Calibration attempt {cal_idx + 1}/{len(jobs)}: {cal_job['name']} ({cal_tokens} tokens)")

        cal_result = run_single_prediction_sequential(cal_job, cal_idx + 1, len(jobs))

        if cal_result['success']:
            completed_jobs.append(cal_result)
            calibration_idx = cal_idx
            cal_peak_vram = cal_result.get('gpu_peak', 0)
            print(f"\n   {chr(0x2705)} Calibration succeeded on job '{cal_job['name']}' -- peak VRAM: {cal_peak_vram:.2f} GB")
            break
        else:
            # Try retry with safer precision for numerical errors
            if cal_result.get('can_retry') and 'nan or inf' in str(cal_result.get('error_messages', [])).lower():
                print(f"\n   {chr(0x1f504)} Retrying '{cal_job['name']}' with safer precision...")
                retried_jobs.append(cal_job['name'])
                clear_gpu_cache()
                time.sleep(2)
                cal_result = run_single_prediction_sequential(cal_job, cal_idx + 1, len(jobs), retry_with_safer=True)
                if cal_result['success']:
                    completed_jobs.append(cal_result)
                    calibration_idx = cal_idx
                    cal_peak_vram = cal_result.get('gpu_peak', 0)
                    print(f"\n   {chr(0x2705)} Calibration succeeded on retry -- peak VRAM: {cal_peak_vram:.2f} GB")
                    break
                else:
                    failed_jobs.append(cal_result)
            else:
                failed_jobs.append(cal_result)

            print(f"   {chr(0x26a0)}  Job '{cal_job['name']}' failed -- trying next job for calibration...")
            clear_gpu_cache()
            time.sleep(2)
            continue

    # Jobs remaining after calibration
    remaining_jobs = jobs[calibration_idx + 1:] if calibration_idx >= 0 else []

    # If NO job succeeded calibration, everything already failed
    if calibration_idx < 0:
        if skipped_in_cal == len(jobs):
            print(f"\n{chr(0x2705)} All {len(jobs)} jobs already completed. Nothing to run.")
        else:
            print(f"\n{chr(0x274c)} ALL {len(jobs)} jobs failed. No successful calibration possible.")
    elif not remaining_jobs:
        print(f"\n{chr(0x2705)} Only one job - calibration complete, nothing more to run.")
    elif cal_peak_vram <= 0:
        # No VRAM data - fall back to sequential
        print(f"\n{chr(0x26a0)}  Could not measure VRAM during calibration. Falling back to sequential.")
        for job_idx, job in enumerate(remaining_jobs, calibration_idx + 2):
            result = run_single_prediction_sequential(job, job_idx, len(jobs))
            if result['success']:
                completed_jobs.append(result)
            else:
                failed_jobs.append(result)
            clear_gpu_cache()
            time.sleep(1)
    elif cal_peak_vram > total_gpu_vram * vram_limit:
        # Single job already uses most of the GPU - sequential only
        print(f"\n{chr(0x26a0)}  Calibration job used {cal_peak_vram:.1f} GB " +
              f"(>{total_gpu_vram * vram_limit:.1f} GB safe limit)")
        print(f"   Single job already fills GPU. Running remaining {len(remaining_jobs)} jobs sequentially.")

        for job_idx, job in enumerate(remaining_jobs, calibration_idx + 2):
            result = run_single_prediction_sequential(job, job_idx, len(jobs))
            if result['success']:
                completed_jobs.append(result)
            else:
                if result.get('can_retry') and 'nan or inf' in str(result.get('error_messages', [])).lower():
                    retried_jobs.append(job['name'])
                    clear_gpu_cache(); time.sleep(2)
                    result = run_single_prediction_sequential(job, job_idx, len(jobs), retry_with_safer=True)
                    if result['success']:
                        completed_jobs.append(result)
                    else:
                        failed_jobs.append(result)
                else:
                    failed_jobs.append(result)
            clear_gpu_cache()
            time.sleep(1)
    else:
        # --- PHASE 2: ESTIMATE REMAINING JOBS ---
        vram_per_token = cal_peak_vram / cal_tokens
        model_overhead = cal_peak_vram * 0.3  # minimum floor

        print(f"\n{chr(0x1f4ca)} PHASE 2: VRAM ESTIMATION")
        print(f"   Calibration: {cal_peak_vram:.2f} GB peak for {cal_tokens} tokens")
        print(f"   Rate: {vram_per_token:.4f} GB/token")
        print(f"   Model overhead floor: {model_overhead:.2f} GB")
        print(f"   Available for parallel: {total_gpu_vram * vram_limit:.1f} GB")

        job_estimates = []
        for job in remaining_jobs:
            tokens = count_job_tokens(job, processor)
            estimated_vram = max(
                vram_per_token * tokens * 1.15,  # 15% safety margin
                model_overhead                    # minimum floor
            )
            job_estimates.append({
                'job': job,
                'tokens': tokens,
                'estimated_vram': estimated_vram
            })
            print(f"   {job['name']}: {tokens} tokens -> ~{estimated_vram:.1f} GB estimated")

        # --- PHASE 3: PARALLEL EXECUTION ---
        print(f"\n{chr(0x1f680)} PHASE 3: PARALLEL EXECUTION")
        safe_limit = total_gpu_vram * vram_limit
        emergency = total_gpu_vram * vram_limit

        # State tracking
        pending_estimates = list(enumerate(job_estimates, calibration_idx + 2))  # (job_number, estimate)
        running_procs = []  # list of dicts with process info
        emergency_triggered = False

        gpu_monitor.start()

        def try_launch_next():
            """Try to launch the next pending job if VRAM allows."""
            if not pending_estimates:
                return False

            current_vram = gpu_monitor.get_current()
            # Use the HIGHER of actual GPU usage or estimated usage of running jobs
            # (processes take time to allocate VRAM, so nvidia-smi lags behind reality)
            estimated_running = sum(p['estimated_vram'] for p in running_procs)
            effective_vram = max(current_vram, estimated_running)
            job_number, est = pending_estimates[0]
            needed = est['estimated_vram']

            if effective_vram + needed < safe_limit:
                pending_estimates.pop(0)
                job = est['job']
                job_name = job['name']
                if config.get('skip_existing', False) and has_existing_output(job_name):
                    print(f"  \u23ed Skipping {job_name}: output already exists")
                    results.append({'name': job_name, 'skipped': True, 'success': True,
                                    'structures': 0, 'time': 0, 'gpu_peak': 0,
                                    'tokens': est['tokens']})
                    return True
                job_dir = get_unique_job_dir(job_name)
                os.makedirs(job_dir, exist_ok=True)

                yaml_content = processor._generate_yaml(job)

                # Inject pre-computed MSA paths if available
                msa_folder_val = global_settings.get('msa_folder', '')
                if msa_folder_val:
                    precomp_msas = find_precomputed_msas(job, msa_folder_val)
                    if precomp_msas:
                        import yaml as _yaml
                        yaml_data = _yaml.safe_load(yaml_content)
                        id_to_name = {}
                        for seq in job.get('sequences', []):
                            if seq.get('type', '').lower() == 'protein' and seq.get('name'):
                                id_to_name[seq.get('id', '')] = seq['name']
                        for entry in yaml_data.get('sequences', []):
                            if 'protein' in entry:
                                protein = entry['protein']
                                chain_id = protein.get('id')
                                ids = chain_id if isinstance(chain_id, list) else [chain_id]
                                for cid in ids:
                                    chain_name = id_to_name.get(cid)
                                    if chain_name and chain_name in precomp_msas:
                                        protein['msa'] = precomp_msas[chain_name]
                                        break
                        yaml_content = _yaml.dump(yaml_data, default_flow_style=False, sort_keys=False)
                yaml_file = os.path.join(job_dir, f"{job_name}.yaml")
                with open(yaml_file, 'w') as f:
                    f.write(yaml_content)

                cmd = build_intellifold_cmd(yaml_file, job_dir, config)

                try:
                    proc = subprocess.Popen(
                        cmd, shell=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, universal_newlines=True
                    )
                except Exception as e:
                    print(f"{chr(0x274c)} Failed to launch {job_name}: {e}")
                    failed_jobs.append({
                        'success': False, 'name': job_name,
                        'error': f'Launch failed: {e}', 'time': 0, 'gpu_peak': 0
                    })
                    return try_launch_next()  # try next

                # Non-blocking reader
                out_q = queue.Queue()
                reader = threading.Thread(target=_reader_thread, args=(proc.stdout, out_q), daemon=True)
                reader.start()

                info = {
                    'process': proc, 'job': job, 'job_name': job_name,
                    'job_dir': job_dir, 'yaml_file': yaml_file,
                    'job_number': job_number, 'tokens': est['tokens'],
                    'estimated_vram': needed, 'start_time': time.time(),
                    'output_queue': out_q, 'error_messages': []
                }
                running_procs.append(info)
                gpu_monitor.register_pid(proc.pid, job_name)

                print(f"\n{chr(0x1f680)} Launched job {job_number}/{len(jobs)}: {job_name} "
                      f"(est. {needed:.1f} GB, used: {effective_vram:.1f}/{safe_limit:.1f} GB)")

                time.sleep(2)  # brief pause for VRAM allocation
                return True
            return False

        # Launch initial jobs
        print(f"\n{chr(0x1f4ca)} Launching parallel jobs...")
        while try_launch_next():
            pass

        if running_procs:
            print(f"\n{chr(0x2705)} {len(running_procs)} job(s) running, {len(pending_estimates)} pending")
        else:
            print(f"\n{chr(0x26a0)}  Could not launch any parallel jobs")

        # Main monitoring loop
        while running_procs or pending_estimates:
            for info in list(running_procs):
                # Drain output queue
                while not info['output_queue'].empty():
                    try:
                        line = info['output_queue'].get_nowait()
                        if not line:
                            continue
                        if 'error' in line.lower() and 'nan or inf' in line.lower():
                            info['error_messages'].append(line)
                        if parse_intellifold_progress(line):
                            cur = gpu_monitor.get_current()
                            print(f"   [{info['job_name']}] {line[:80]} [GPU: {cur:.1f} GB]")
                        elif 'error' in line.lower() or 'warning' in line.lower():
                            print(f"   {chr(0x26a0)} [{info['job_name']}] {line[:80]}")
                    except queue.Empty:
                        break

                # Check completion
                rc = info['process'].poll()
                if rc is not None:
                    running_procs.remove(info)
                    job_time = time.time() - info['start_time']
                    pid_peak = gpu_monitor.unregister_pid(info['process'].pid)

                    # Final drain: capture any lines buffered between last poll and exit
                    while not info['output_queue'].empty():
                        try:
                            info['output_queue'].get_nowait()
                        except queue.Empty:
                            break

                    try:
                        if rc == 0:
                            print(f"\n{chr(0x2705)} Completed: {info['job_name']} ({job_time:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            result = process_completed_job(
                                info['job_name'], info['job_dir'], info['yaml_file'],
                                job_time, pid_peak, drive, gdrive_folder_id
                            )
                            result['tokens'] = info['tokens']
                            if result['success']:
                                completed_jobs.append(result)
                            else:
                                failed_jobs.append(result)
                        else:
                            print(f"\n{chr(0x274c)} Failed: {info['job_name']} (exit code {rc}, {job_time:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            error_files = read_error_files(info['job_dir'], info['job_name'])
                            if error_files:
                                for ef in error_files[:1]:
                                    print(f"   {ef['content'][:200]}")
                            failed_jobs.append({
                                'success': False, 'name': info['job_name'],
                                'error': f'Exit code {rc}', 'error_messages': info['error_messages'],
                                'time': job_time, 'gpu_peak': pid_peak
                            })
                    except Exception as e:
                        print(f"  ERROR processing {info['job_name']}: {e}")
                        failed_jobs.append({
                            'success': False, 'name': info['job_name'],
                            'error': str(e), 'time': job_time, 'gpu_peak': pid_peak
                        })

                    # Try launching next
                    clear_gpu_cache()
                    time.sleep(1)
                    gpu_monitor.reset_peak()

                    if not emergency_triggered:
                        while try_launch_next():
                            pass
                        if running_procs or pending_estimates:
                            print(f"   {chr(0x1f4ca)} Status: {len(running_procs)} running, "
                                  f"{len(pending_estimates)} pending, "
                                  f"{len(completed_jobs)} done, {len(failed_jobs)} failed")

            # Emergency VRAM check
            current_vram = gpu_monitor.get_current()
            if current_vram > emergency and not emergency_triggered:
                emergency_triggered = True
                print(f"\n{chr(0x1f6a8)} EMERGENCY: GPU VRAM at {current_vram:.1f}/{total_gpu_vram:.1f} GB")
                print(f"   Stopping new launches. Waiting for {len(running_procs)} running jobs...")

            time.sleep(2)

        peak_vram = gpu_monitor.stop()

        # If there are remaining jobs after emergency, run sequentially
        if pending_estimates:
            print(f"\n{chr(0x1f504)} Running {len(pending_estimates)} remaining jobs sequentially after emergency...")
            for job_number, est in pending_estimates:
                result = run_single_prediction_sequential(est['job'], job_number, len(jobs))
                if result['success']:
                    completed_jobs.append(result)
                else:
                    failed_jobs.append(result)
                clear_gpu_cache()
                time.sleep(1)


# ============================================================
# FINAL ipSAE BATCH ANALYSIS
# ============================================================
ipsae_batch_url = None
if (ipsae_available and config.get('enable_ipsae')
        and config.get('ipsae_batch_analysis') and len(completed_jobs) > 1):
    print("\n" + "=" * 60)
    print("FINAL ipSAE BATCH ANALYSIS")
    print("=" * 60)

    ipsae_batch_output = "/content/ipsae_batch_results"
    os.makedirs(ipsae_batch_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", "/content",
        "--backend", "intellifold",
        "--output_dir", ipsae_batch_output,
        "--pae_cutoff", str(config.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(config.get('ipsae_dist_cutoff', 10.0)),
        "--cores", str(os.cpu_count() or 2),
    ]
    if config.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if config.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if config.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", config['ipsae_select_residues']])

    print(f"Analyzing {len(completed_jobs)} completed jobs...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=600)
        if result.returncode == 0:
            batch_files = [f for f in os.listdir(ipsae_batch_output) if not f.startswith('.')]
            print(f"Batch analysis complete: {len(batch_files)} files")
            for bf in sorted(batch_files):
                size = os.path.getsize(os.path.join(ipsae_batch_output, bf))
                print(f"   {bf} ({size//1024}KB)")

            # Upload to GDrive
            if drive and gdrive_folder_id:
                batch_zip = "/content/ipsae_batch_results.zip"
                with zipfile.ZipFile(batch_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
                    for root, dirs, files_list in os.walk(ipsae_batch_output):
                        for f in files_list:
                            fp = os.path.join(root, f)
                            arcname = os.path.join("ipsae_batch_results", os.path.relpath(fp, ipsae_batch_output))
                            zipf.write(fp, arcname)
                ipsae_batch_url = upload_to_gdrive(drive, batch_zip, gdrive_folder_id, "ipsae_batch")
                print(f"Uploaded: {ipsae_batch_url}")
            else:
                print(f"Saved locally: {ipsae_batch_output}/")
        else:
            print(f"Batch analysis failed (rc={result.returncode})")
            if result.stderr:
                print(f"   {result.stderr[:300]}")
    except subprocess.TimeoutExpired:
        print("Batch analysis timed out (600s)")
    except Exception as e:
        print(f"Batch analysis error: {e}")

# ============================================================
# FINAL SUMMARY
# ============================================================
total_time = time.time() - batch_start_time

print("\n" + "=" * 60)
print(f"{chr(0x1f389)} BATCH PROCESSING COMPLETE")
print("=" * 60)
print(f"Ended: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total duration: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"\nCompleted: {len(completed_jobs)}/{len(jobs)} jobs")
print(f"Failed: {len(failed_jobs)}/{len(jobs)} jobs")
if retried_jobs:
    print(f"Retried with safer settings: {len(retried_jobs)} jobs")

if completed_jobs:
    print(f"\n{chr(0x2705)} Successful jobs:")
    for job in completed_jobs:
        time_str = f"{job['time']:.1f}s" if job['time'] < 60 else f"{job['time']/60:.1f}m"
        gpu_str = f" | Peak GPU: {job['gpu_peak']:.2f} GB" if job.get('gpu_peak') else ""
        structs = job.get('structures', '?')
        print(f"  - {job['name']}: {structs} structures ({time_str}{gpu_str})")
        if job.get('url'):
            print(f"    {chr(0x1f517)} {job['url']}")

if failed_jobs:
    print(f"\n{chr(0x274c)} Failed jobs:")
    for job in failed_jobs:
        error = job.get('error', 'Unknown error')[:80]
        time_str = f"{job['time']:.1f}s" if job.get('time', 0) < 60 else f"{job.get('time', 0)/60:.1f}m"
        print(f"  - {job['name']}: {error} ({time_str})")

# GPU statistics
if gpu_available and completed_jobs:
    gpu_peaks = [j['gpu_peak'] for j in completed_jobs if j.get('gpu_peak') and j['gpu_peak'] > 0]
    if gpu_peaks:
        print(f"\n{chr(0x1f3ae)} GPU Peak Statistics:")
        print(f"   Average: {sum(gpu_peaks)/len(gpu_peaks):.2f} GB")
        print(f"   Highest: {max(gpu_peaks):.2f} GB")
        print(f"   Lowest: {min(gpu_peaks):.2f} GB")
        print(f"   Capacity: {total_gpu_vram:.1f} GB")
        print(f"   Peak utilization: {(max(gpu_peaks)/total_gpu_vram)*100:.1f}%")


# ipSAE Analysis Summary
if config.get('enable_ipsae'):
    ipsae_count = sum(1 for j in completed_jobs if j.get('ipsae_ran'))
    print(f"\nipSAE Interface Analysis:")
    print(f"   Per-job: {ipsae_count}/{len(completed_jobs)} jobs analyzed")
    if ipsae_batch_url:
        print(f"   Batch comparison: {ipsae_batch_url}")

if completed_jobs:
    avg_time = sum(j['time'] for j in completed_jobs) / len(completed_jobs)
    print(f"\n{chr(0x23f1)}  Average time per job: {avg_time:.1f}s ({avg_time/60:.1f} min)")

if drive and gdrive_folder_id:
    print(f"\n{chr(0x2601)}  All results uploaded to: {gdrive_folder_name}")
    print(f"   {chr(0x1f4c1)} https://drive.google.com/drive/folders/{gdrive_folder_id}")
else:
    print(f"\n{chr(0x1f4c1)} All results saved locally in job directories")

print(f"\n{chr(0x1f389)} All done!")
